# Hidden Markov Models (HMM)

We will evaluate multiple HMM models for the purpose of Regime detection over Bitcoin prices. There will be a combination of features that will be inputed in our models. This notebook will cover HMM with skewed-t distribution, HMM with rolling window, Non homogeneous HMM, HMM with GMM, and HMM with Factor analysis. There is only the DNN HMM model which we will evaluate in another notebook. We will evaluate all models using silhouette scores, log-likelihood, and plots.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import itertools
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import FactorAnalysis, PCA
from hmmlearn.base import _BaseHMM
from scipy import stats
from scipy.special import logsumexp
from joblib import Parallel, delayed
import multiprocessing
import numba
from tqdm import tqdm
import os
import json
from time import time
sns.set(style='whitegrid')

## Data loading / pre processing

In [2]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
train_price = pd.read_csv("train_price.csv", index_col="Date", parse_dates=True)['BTC-USD']
test_price = pd.read_csv("test_price.csv", index_col="Date", parse_dates=True)['BTC-USD']

In [3]:
# Remove the first two rows and reset the index
X_train = X_train.iloc[2:].reset_index(drop=True)
# Rename the "Price" column to "Date"
X_train.rename(columns={"Price": "Date"}, inplace=True)
# Convert the "Date" column to datetime format
X_train["Date"] = pd.to_datetime(X_train["Date"])
# Set "Date" as the index
X_train.set_index("Date", inplace=True)
# Remove the first two rows and reset the index
X_test = X_test.iloc[2:].reset_index(drop=True)
# Rename the "Price" column to "Date"
X_test.rename(columns={"Price": "Date"}, inplace=True)
# Convert the "Date" column to datetime format
X_test["Date"] = pd.to_datetime(X_test["Date"])
# Set "Date" as the index
X_test.set_index("Date", inplace=True)

In [4]:
X_train.head()

,Returns,Range,Log_Range,pct_Range,fg,pct_fg,RSI,pct_RSI
Date,,,,,,,,
2018-02-16,0.006618,0.050818,0.049569,-0.431059,67.0,-0.056338,63.799058,0.007788
2018-02-17,0.082383,0.097553,0.093083,0.919639,74.0,0.104478,69.537696,0.089949
2018-02-18,-0.051792,0.099148,0.094535,0.016353,63.0,-0.148649,62.705291,-0.098255
2018-02-19,0.061874,0.072347,0.069850,-0.270310,67.0,0.063492,66.909576,0.067048
2018-02-20,0.015768,0.064700,0.062693,-0.105697,74.0,0.104478,67.940541,0.015408


## Data Scaling 

In [5]:
scaler = RobustScaler()
X_train_scaled_array = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled_array, index=X_train.index, columns=X_train.columns)

X_test_scaled_array = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled_array, index=X_test.index, columns=X_test.columns)

## Feature combinatios

In [6]:
# features = []
# for r in range(1, len(X_train.columns) + 1):
#     features.extend(combinations(X_train.columns, r))
test_col = ['Range', 'fg', 'RSI']
features = []

# Create all possible combinations of 1, 2, and 3 features
for r in range(1, len(test_col) + 1):
    # Create combinations and convert each tuple to a list
    combinations = [list(combo) for combo in itertools.combinations(test_col, r)]
    features.extend(combinations)

print(features)

[['Range'], ['fg'], ['RSI'], ['Range', 'fg'], ['Range', 'RSI'], ['fg', 'RSI'], ['Range', 'fg', 'RSI']]


In [7]:
X_train = X_train[test_col]
X_test_scaled = X_test_scaled[test_col]

### BaseHMM

In [8]:
from abc import ABC, abstractmethod

class BaseHMMModel(ABC):
    """
    Optimized abstract base class for Hidden Markov Models with standardized interface.
    Handles training, evaluation, and storage of HMM models with different feature combinations.
    """
    
    def __init__(self, n_states=3, model_name="BaseHMM", n_jobs=1):
        """
        Initialize the OptimizedBaseHMMModel.
        
        Parameters:
        -----------
        n_states : int
            Number of hidden states/regimes
        model_name : str
            Name identifier for this model type
        n_jobs : int
            Number of parallel jobs for feature combination processing
            Default: 1 (no parallelization), -1 means use all processors
        """
        self.n_states = n_states
        self.model_name = model_name
        self.models_info = {}  # Dictionary to store all model results
        self.model_id_counter = 0  # Counter for generating unique model IDs
        self.n_jobs = n_jobs
        
        # Add checkpointing variables 
        self.checkpoint_dir = "./hmm_checkpoints"
        os.makedirs(self.checkpoint_dir, exist_ok=True)
    
    def _process_feature_combination(self, X_train, X_train_scaled, feature_tuple, is_scaled):
        """
        Process a single feature combination and scaling option.
        Designed to be called in parallel for different feature combinations.
        
        Parameters:
        -----------
        X_train : pandas.DataFrame
            Raw training data
        X_train_scaled : pandas.DataFrame
            Scaled training data
        feature_tuple : tuple or list
            Feature combination to use
        is_scaled : bool
            Whether to use scaled data
            
        Returns:
        --------
        tuple
            (model_info, success_flag)
        """
        # Select appropriate data
        X = X_train_scaled if is_scaled else X_train
        scaling_name = "robust" if is_scaled else "none"
        
        # Convert tuple to list if needed
        feature_list = list(feature_tuple) if isinstance(feature_tuple, tuple) else feature_tuple
        feature_name = "+".join(feature_list)
        
        try:
            # Extract data for this feature combination
            X_subset = X[feature_list].values
            
            # Track time for this model
            start_time = time()
            
            # Train the model
            model, log_likelihood, regimes = self._train_and_predict(X_subset, is_scaled)
            
            # Calculate processing time
            processing_time = time() - start_time
            
            # Calculate silhouette score (handle single feature case)
            silhouette = self._calculate_silhouette(X_subset, regimes)
            
            # Return model information (without assigning an ID yet)
            return {
                'model': model,
                'model_type': self.model_name,
                'scaling': scaling_name,
                'features': feature_list,
                'silhouette_score': silhouette,
                'log_likelihood': log_likelihood,
                'regimes': regimes,
                'processing_time': processing_time
            }, True
            
        except Exception as e:
            print(f"Error training model with features {feature_list}, scaling {scaling_name}: {str(e)}")
            return None, False
    
    def train_all_models(self, X_train, X_train_scaled, features, batch_size=None, checkpoint_frequency=5):
        """
        Train models on all feature combinations with both scaled and unscaled data.
        Uses parallel processing and checkpointing for better performance.
        
        Parameters:
        -----------
        X_train : pandas.DataFrame
            Raw training data
        X_train_scaled : pandas.DataFrame
            Scaled training data (e.g., using RobustScaler)
        features : list of tuples
            List of feature combinations to use, each item is a tuple of feature names
        batch_size : int or None
            Size of batches for processing, None means process all at once
        checkpoint_frequency : int
            How often to save checkpoints (in number of batches)
            
        Returns:
        --------
        dict
            Updated models_info dictionary with all trained models
        """
        # Check for existing checkpoint - using JSON for features
        # Note: Model data can't be stored in JSON, so we'll only checkpoint processed features
        features_path = os.path.join(self.checkpoint_dir, f"{self.model_name}_features_processed.json")
        
        processed_features = set()
        
        # Try to load checkpoint if it exists
        if os.path.exists(features_path):
            try:
                with open(features_path, 'r') as f:
                    # Convert saved list back to set of tuples
                    processed_list = json.load(f)
                    processed_features = set(tuple(item) if isinstance(item[0], list) else tuple(item) 
                                            for item in processed_list)
                
                print(f"Loaded checkpoint with {len(processed_features)} processed features")
            except Exception as e:
                print(f"Error loading checkpoint: {str(e)}. Starting fresh.")
        
        # Create all combinations of features and scaling options
        tasks = []
        for feature in features:
            feature_key = tuple(feature) if isinstance(feature, list) else feature
            
            for is_scaled in [False, True]:
                # Skip if already processed
                task_key = (feature_key, is_scaled)
                if task_key in processed_features:
                    continue
                
                tasks.append((feature, is_scaled))
        
        if not tasks:
            print("All feature combinations already processed.")
            return self.models_info
        
        # Process in batches if specified
        if batch_size is None:
            batch_size = len(tasks)
        
        num_batches = (len(tasks) + batch_size - 1) // batch_size
        
        for batch_idx in range(num_batches):
            batch_start = batch_idx * batch_size
            batch_end = min((batch_idx + 1) * batch_size, len(tasks))
            batch_tasks = tasks[batch_start:batch_end]
            
            print(f"\nProcessing batch {batch_idx + 1}/{num_batches} with {len(batch_tasks)} tasks")
            
            # Group tasks by scaling for more efficient processing
            unscaled_tasks = [task[0] for task in batch_tasks if not task[1]]
            scaled_tasks = [task[0] for task in batch_tasks if task[1]]
            
            # Process unscaled features
            if unscaled_tasks:
                print(f"Training {self.model_name} models with none scaling")
                self._process_feature_group(X_train, X_train_scaled, unscaled_tasks, False, processed_features)
            
            # Process scaled features
            if scaled_tasks:
                print(f"Training {self.model_name} models with robust scaling")
                self._process_feature_group(X_train, X_train_scaled, scaled_tasks, True, processed_features)
            
            # Save checkpoint if needed - only saving processed features
            if (batch_idx + 1) % checkpoint_frequency == 0 or batch_idx == num_batches - 1:
                # Convert set of tuples to list for JSON serialization
                processed_list = [list(item) for item in processed_features]
                
                with open(features_path, 'w') as f:
                    json.dump(processed_list, f)
                
                print(f"Saved checkpoint after batch {batch_idx + 1}. {len(processed_features)} features processed.")
        
        return self.models_info
    
    def _process_feature_group(self, X_train, X_train_scaled, feature_group, is_scaled, processed_features):
        """
        Process a group of features with the same scaling option.
        
        Parameters:
        -----------
        X_train : pandas.DataFrame
            Raw training data
        X_train_scaled : pandas.DataFrame
            Scaled training data
        feature_group : list of tuples
            Features to process
        is_scaled : bool
            Whether to use scaled data
        processed_features : set
            Set of already processed features to update
            
        Returns:
        --------
        None (updates self.models_info directly)
        """
        scaling_name = "robust" if is_scaled else "none"
        
        # Process in parallel if n_jobs is not 1
        if self.n_jobs != 1 and len(feature_group) > 1:
            # Create parallel tasks
            results = Parallel(n_jobs=self.n_jobs)(
                delayed(self._process_feature_combination)(
                    X_train, X_train_scaled, feature, is_scaled
                ) for feature in tqdm(feature_group, desc=f"Training models ({scaling_name} scaling)")
            )
            
            # Process results
            for feature, (model_info, success) in zip(feature_group, results):
                feature_key = tuple(feature) if isinstance(feature, list) else feature
                task_key = (feature_key, is_scaled)
                
                if success and model_info is not None:
                    # Generate unique ID
                    feature_str = "+".join(model_info['features'])
                    model_id = f"{self.model_name}_{scaling_name}_{feature_str}_{self.model_id_counter}"
                    self.model_id_counter += 1
                    
                    # Store model
                    self.models_info[model_id] = model_info
                
                # Mark as processed
                processed_features.add(task_key)
        else:
            # Process sequentially
            for feature in tqdm(feature_group, desc=f"Training models ({scaling_name} scaling)"):
                feature_key = tuple(feature) if isinstance(feature, list) else feature
                task_key = (feature_key, is_scaled)
                
                model_info, success = self._process_feature_combination(
                    X_train, X_train_scaled, feature, is_scaled
                )
                
                if success and model_info is not None:
                    # Generate unique ID
                    feature_str = "+".join(model_info['features'])
                    model_id = f"{self.model_name}_{scaling_name}_{feature_str}_{self.model_id_counter}"
                    self.model_id_counter += 1
                    
                    # Store model
                    self.models_info[model_id] = model_info
                
                # Mark as processed
                processed_features.add(task_key)
    
    def _calculate_silhouette(self, X_subset, regimes):
        """
        Calculate silhouette score or alternative metric for single feature case.
        Optimized for performance.
        
        Parameters:
        -----------
        X_subset : numpy.ndarray
            Feature data subset
        regimes : numpy.ndarray
            Predicted regimes from the model
            
        Returns:
        --------
        float or None
            Silhouette score or alternative metric, None if calculation fails
        """
        # Quick checks before expensive computation
        # Check if we have enough unique regimes
        unique_regimes = np.unique(regimes)
        if len(unique_regimes) <= 1:
            return 0.0
        
        # Special handling for 1D data
        n_features = X_subset.shape[1] if X_subset.ndim > 1 else 1
        
        # For multi-feature data, try standard silhouette score
        if n_features > 1:
            try:
                return silhouette_score(X_subset, regimes)
            except Exception as e:
                print(f"Silhouette score calculation failed: {str(e)}")
                return None
        else:
            # For 1D data, use simplified variance ratio calculation
            try:
                # Reshape 1D data if needed
                X_flat = X_subset.flatten() if X_subset.ndim > 1 else X_subset
                
                # Get overall mean
                overall_mean = np.mean(X_flat)
                overall_var = np.var(X_flat)
                
                if overall_var < 1e-10:  # Effectively zero variance
                    return 0.0
                
                # Calculate between and within variance
                between_var = 0
                within_var = 0
                
                # Calculate weighted means and variances for each regime
                for regime in unique_regimes:
                    mask = (regimes == regime)
                    count = np.sum(mask)
                    if count > 0:
                        regime_data = X_flat[mask]
                        regime_mean = np.mean(regime_data)
                        regime_var = np.var(regime_data)
                        
                        # Update between variance
                        between_var += count * (regime_mean - overall_mean) ** 2
                        
                        # Update within variance
                        within_var += count * regime_var
                
                # Normalize by sample size
                between_var /= len(X_flat)
                within_var /= len(X_flat)
                
                # Handle edge cases
                if within_var < 1e-10:  # Effectively zero within-variance
                    return 1.0 if between_var > 0 else 0.0
                
                # Calculate ratio and transform to [-1, 1] range
                ratio = between_var / (between_var + within_var)
                return 2 * ratio - 1
                
            except Exception as e:
                print(f"Alternative silhouette calculation failed: {str(e)}")
                return None
    
    def predict_with_model(self, model_id, X_new):
        """
        Make predictions with a specific model.
        
        Parameters:
        -----------
        model_id : str
            ID of the model to use for prediction
        X_new : pandas.DataFrame
            New data for prediction
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        if model_id not in self.models_info:
            raise ValueError(f"Model ID {model_id} not found")
        
        # Get model information
        model_info = self.models_info[model_id]
        model = model_info['model']
        features = model_info['features']
        
        # Extract features for prediction
        X_subset = X_new[features].values
        
        # Make prediction using the specific model's prediction logic
        return self._predict(model, X_subset)
    
    def get_best_models(self, metric='silhouette_score', top_n=5, min_features=1, max_features=None):
        """
        Get the best performing models based on a specified metric.
        
        Parameters:
        -----------
        metric : str
            Metric to use for ranking ('silhouette_score' or 'log_likelihood')
        top_n : int
            Number of top models to return
        min_features : int
            Minimum number of features
        max_features : int or None
            Maximum number of features, None means no limit
            
        Returns:
        --------
        list
            List of (model_id, score) tuples sorted by score
        """
        # Filter models by feature count
        filtered_models = []
        
        for model_id, info in self.models_info.items():
            feature_count = len(info['features'])
            if feature_count >= min_features and (max_features is None or feature_count <= max_features):
                # Check if the metric exists and is not None
                if metric in info and info[metric] is not None:
                    filtered_models.append((model_id, info[metric]))
        
        # Sort by metric (higher is better for silhouette, log_likelihood)
        sorted_models = sorted(filtered_models, key=lambda x: x[1], reverse=True)
        
        # Return top N models
        return sorted_models[:top_n]
    
    def analyze_feature_importance(self):
        """
        Analyze which features appear most frequently in the top performing models.
        
        Returns:
        --------
        dict
            Dictionary of feature importance scores
        """
        # Get models with valid silhouette scores
        valid_models = {
            model_id: info for model_id, info in self.models_info.items()
            if info['silhouette_score'] is not None
        }
        
        if not valid_models:
            return {}
        
        # Get unique features across all models
        all_features = set()
        for info in valid_models.values():
            all_features.update(info['features'])
        
        # Calculate feature importance
        feature_importance = {feature: 0.0 for feature in all_features}
        
        # Weight features by their silhouette score in models
        for model_id, info in valid_models.items():
            score = info['silhouette_score']
            for feature in info['features']:
                feature_importance[feature] += score
        
        # Normalize by number of models
        for feature in feature_importance:
            feature_importance[feature] /= len(valid_models)
        
        # Sort by importance (descending)
        return dict(sorted(feature_importance.items(), key=lambda x: x[1], reverse=True))
    
    def save_model_metadata(self, filename):
        """
        Save model metadata (without the actual model objects) to a JSON file.
        Useful for analyzing results without loading the full models.
        
        Parameters:
        -----------
        filename : str
            File path to save the metadata
            
        Returns:
        --------
        bool
            Success status
        """
        try:
            # Create serializable metadata dictionary
            metadata = {}
            
            for model_id, info in self.models_info.items():
                # Extract only serializable metadata
                metadata[model_id] = {
                    'model_type': info['model_type'],
                    'scaling': info['scaling'],
                    'features': info['features'],
                    'silhouette_score': float(info['silhouette_score']) if info['silhouette_score'] is not None else None,
                    'log_likelihood': float(info['log_likelihood']) if info['log_likelihood'] is not None else None,
                    'processing_time': float(info.get('processing_time', 0))
                }
            
            with open(filename, 'w') as f:
                json.dump(metadata, f, indent=2)
            
            return True
        except Exception as e:
            print(f"Error saving model metadata: {str(e)}")
            return False
    
    def export_results(self, filename):
        """
        Export key results in a format suitable for analysis.
        Creates a CSV file with model results.
        
        Parameters:
        -----------
        filename : str
            File path to save the results
            
        Returns:
        --------
        bool
            Success status
        """
        try:
            import pandas as pd
            
            # Create results dataframe
            results = []
            
            for model_id, info in self.models_info.items():
                # Extract basic information
                result = {
                    'model_id': model_id,
                    'model_type': info['model_type'],
                    'scaling': info['scaling'],
                    'features': '+'.join(info['features']),
                    'feature_count': len(info['features']),
                    'silhouette_score': info['silhouette_score'],
                    'log_likelihood': info['log_likelihood'],
                    'processing_time': info.get('processing_time', None)
                }
                
                results.append(result)
            
            # Convert to dataframe and save
            df = pd.DataFrame(results)
            df.to_csv(filename, index=False)
            
            print(f"Exported {len(results)} model results to {filename}")
            return True
        except Exception as e:
            print(f"Error exporting results: {str(e)}")
            return False
    
    @abstractmethod
    def _train_and_predict(self, X_subset, is_scaled):
        """
        Train model and generate predictions for the given data.
        Must be implemented by subclasses.
        
        Parameters:
        -----------
        X_subset : numpy.ndarray
            Feature data subset
        is_scaled : bool
            Whether the data is already scaled
            
        Returns:
        --------
        tuple
            (trained_model, log_likelihood, regimes)
        """
        pass
    
    @abstractmethod
    def _predict(self, model, X_subset):
        """
        Make predictions with the trained model.
        Must be implemented by subclasses.
        
        Parameters:
        -----------
        model : object
            Trained model
        X_subset : numpy.ndarray
            Feature data subset
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        pass

## StandardHMM model

In [9]:
from hmmlearn import hmm

class StandardHMM(BaseHMMModel):
    """
    Standard Gaussian HMM implementation using hmmlearn.
    """
    
    def __init__(self, n_states=3):
        super().__init__(n_states=n_states, model_name="StandardHMM")
        
    def _train_and_predict(self, X_subset, is_scaled):
        """
        Train model and generate predictions for the given data.
        
        Parameters:
        -----------
        X_subset : numpy.ndarray
            Feature data subset
        is_scaled : bool
            Whether the data is already scaled
            
        Returns:
        --------
        tuple
            (trained_model, log_likelihood, regimes)
        """
        # Debug output
        print(f"  Training with X_subset shape: {X_subset.shape}")
        
        # Handle reshaping properly
        if X_subset.ndim == 1:
            X_hmm = X_subset.reshape(-1, 1)
            n_features = 1
        else:
            n_features = X_subset.shape[1]
            if n_features == 1:
                X_hmm = X_subset  # Already 2D with shape (n_samples, 1)
            else:
                X_hmm = X_subset  # Multi-feature, no reshape needed
        
        # Choose appropriate covariance type
        if n_features == 1:
            cov_type = "diag"  # Diagonal for 1D data
        else:
            # Use full if we have enough samples, otherwise diagonal
            n_samples = X_hmm.shape[0]
            if n_samples >= n_features * 10:
                cov_type = "full"
            else:
                cov_type = "diag"
        
        # Create and fit the model
        try:
            model = hmm.GaussianHMM(
                n_components=self.n_states,
                covariance_type=cov_type,
                n_iter=100,
                random_state=42
            )
            
            # Check data for NaN or inf values
            if np.isnan(X_hmm).any() or np.isinf(X_hmm).any():
                print(f"  Warning: Data contains NaN or inf values. Replacing with zeros.")
                X_hmm = np.nan_to_num(X_hmm)
            
            # Fit the model
            model.fit(X_hmm)
            
            # Get log likelihood
            log_likelihood = model.score(X_hmm)
            
            # Get state predictions
            regimes = model.predict(X_hmm)
            
            return model, log_likelihood, regimes
            
        except Exception as e:
            print(f"  Error in training model: {str(e)}")
            # Re-raise the exception to be caught by the caller
            raise
    
    def _predict(self, model, X_subset):
        """
        Make predictions with the trained model.
        
        Parameters:
        -----------
        model : hmm.GaussianHMM
            Trained model
        X_subset : numpy.ndarray
            Feature data subset
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        # Handle reshaping properly
        if X_subset.ndim == 1:
            X_hmm = X_subset.reshape(-1, 1)
        else:
            X_hmm = X_subset
            
        # Check data for NaN or inf values
        if np.isnan(X_hmm).any() or np.isinf(X_hmm).any():
            X_hmm = np.nan_to_num(X_hmm)
            
        try:
            return model.predict(X_hmm)
        except Exception as e:
            print(f"  Error in prediction: {str(e)}")
            # Return a default prediction (all zeros) in case of error
            return np.zeros(X_hmm.shape[0], dtype=int)

### HMM with a Skew-t distribution

In [10]:
class SkewTHMM(BaseHMMModel):
    """
    Optimized Hidden Markov Model with Skew-t distribution for observations.
    Suitable for financial data with skewness and heavy tails.
    """
    
    def __init__(self, n_states=3, max_iter=30, tol=1e-4, n_jobs=-1):
        """
        Initialize the OptimizedSkewTHMM model.
        
        Parameters:
        -----------
        n_states : int
            Number of hidden states/regimes
        max_iter : int
            Maximum number of EM iterations
        tol : float
            Convergence tolerance
        n_jobs : int
            Number of jobs for parallel processing. -1 means using all processors.
        """
        super().__init__(n_states=n_states, model_name="SkewTHMM")
        self.max_iter = max_iter
        self.tol = tol
        self.n_jobs = n_jobs
    
    def _train_and_predict(self, X_subset, is_scaled):
        """
        Train SkewTHMM model and generate predictions.
        
        Parameters:
        -----------
        X_subset : numpy.ndarray
            Feature data subset
        is_scaled : bool
            Whether the data is already scaled
            
        Returns:
        --------
        tuple
            (trained_model, log_likelihood, regimes)
        """
        # Initialize model parameters
        model = self._create_model(X_subset.shape[1], is_scaled)
        
        # Fit the model
        model = self._fit_model(model, X_subset)
        
        # Generate predictions
        regimes = model['states']
        
        # Calculate log likelihood
        log_likelihood = model['log_likelihood']
        
        return model, log_likelihood, regimes
    
    def _predict(self, model, X_subset):
        """
        Make predictions with the trained SkewTHMM model using Viterbi algorithm.
        
        Parameters:
        -----------
        model : dict
            Trained model parameters
        X_subset : numpy.ndarray
            Feature data subset
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        # Apply scaling if needed and if model has a scaler
        if model.get('scaler') is not None:
            X_scaled = model['scaler'].transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Pre-compute all emission log probabilities
        n_samples = X_scaled.shape[0]
        n_states = self.n_states
        
        # Parallelize emission probability calculation
        if self.n_jobs != 1:
            log_emissions = self._calculate_all_emissions_parallel(X_scaled, model)
        else:
            log_emissions = np.zeros((n_samples, n_states))
            for i in range(n_states):
                for t in range(n_samples):
                    log_emissions[t, i] = self._emission_log_prob_vectorized(X_scaled[t], model, i)
        
        # Initialize Viterbi variables
        delta = np.zeros((n_samples, n_states))
        psi = np.zeros((n_samples, n_states), dtype=int)
        
        # Log of initial probabilities
        log_init_probs = np.log(model['initial_probs'])
        
        # Log of transition matrix
        log_trans_mat = np.log(model['transition_matrix'])
        
        # Initialization
        delta[0] = log_init_probs + log_emissions[0]
        
        # Recursion (Viterbi algorithm)
        for t in range(1, n_samples):
            for j in range(n_states):
                probs = delta[t-1] + log_trans_mat[:, j]
                psi[t, j] = np.argmax(probs)
                delta[t, j] = probs[psi[t, j]] + log_emissions[t, j]
        
        # Backtracking
        states = np.zeros(n_samples, dtype=int)
        states[-1] = np.argmax(delta[-1])
        
        for t in range(n_samples-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        
        return states
    
    def _calculate_all_emissions_parallel(self, X, model):
        """
        Calculate all emission log probabilities in parallel.
        
        Parameters:
        -----------
        X : numpy.ndarray
            Feature data
        model : dict
            Model parameters
            
        Returns:
        --------
        numpy.ndarray
            Log emission probabilities matrix
        """
        n_samples = X.shape[0]
        n_states = model['n_states']
        
        # Define a function to calculate emissions for a given state
        def calc_emissions_for_state(state_idx):
            emissions = np.zeros(n_samples)
            for t in range(n_samples):
                emissions[t] = self._emission_log_prob_vectorized(X[t], model, state_idx)
            return emissions
        
        # Calculate emissions for all states in parallel
        results = Parallel(n_jobs=self.n_jobs)(
            delayed(calc_emissions_for_state)(i) for i in range(n_states)
        )
        
        # Combine results
        log_emissions = np.zeros((n_samples, n_states))
        for i, emissions in enumerate(results):
            log_emissions[:, i] = emissions
        
        return log_emissions
    
    def _create_model(self, n_features, is_scaled):
        """
        Create a new SkewTHMM model with initialized parameters.
        
        Parameters:
        -----------
        n_features : int
            Number of features in the data
        is_scaled : bool
            Whether the data is already scaled
            
        Returns:
        --------
        dict
            Model parameters dictionary
        """
        # Initialize model parameters
        model = {
            'n_states': self.n_states,
            'n_features': n_features,
            'initial_probs': np.ones(self.n_states) / self.n_states,
            'transition_matrix': np.ones((self.n_states, self.n_states)) / self.n_states,
            'location': np.zeros((self.n_states, n_features)),
            'scale': np.ones((self.n_states, n_features)),
            'shape': np.zeros((self.n_states, n_features)),  # skewness parameter
            'df': 4 * np.ones((self.n_states, n_features)),  # degrees of freedom
            'scaler': None if is_scaled else RobustScaler(),
            'log_likelihood': -np.inf,
            'states': None
        }
        
        return model
    
    def _fit_model(self, model, X_subset):
        """
        Fit the SkewTHMM model using EM algorithm with optimizations.
        
        Parameters:
        -----------
        model : dict
            Model parameters dictionary
        X_subset : numpy.ndarray
            Feature data subset
            
        Returns:
        --------
        dict
            Updated model parameters
        """
        n_samples, n_features = X_subset.shape
        
        # Apply scaling if needed
        if model['scaler'] is not None:
            X_scaled = model['scaler'].fit_transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Initialize using k-means with limited iterations for speed
        kmeans = KMeans(n_clusters=model['n_states'], 
                        random_state=42, 
                        n_init=3,  # Reduce number of initializations
                        max_iter=20)  # Limit iterations
        states = kmeans.fit_predict(X_scaled)
        
        # Initialize parameters based on clusters
        for i in range(model['n_states']):
            mask = (states == i)
            if np.sum(mask) > 0:
                # Location (mean)
                model['location'][i] = np.median(X_scaled[mask], axis=0)
                
                # Scale (spread)
                model['scale'][i] = np.median(np.abs(X_scaled[mask] - model['location'][i]), axis=0)
                model['scale'][i] = np.maximum(model['scale'][i], 1e-6)  # Prevent zero scale
                
                # Shape (skewness) - simplified calculation
                if np.sum(mask) > 10:
                    model['shape'][i] = stats.skew(X_scaled[mask], axis=0)
                
                # Degrees of freedom (tail heaviness) - simplified calculation
                if np.sum(mask) > 20:
                    kurtosis = stats.kurtosis(X_scaled[mask], axis=0)
                    model['df'][i] = np.maximum(4, np.minimum(30, 30 / (1 + np.abs(kurtosis))))
        
        # Estimate initial state probabilities
        for i in range(model['n_states']):
            model['initial_probs'][i] = np.mean(states == i)
        
        # Estimate transition matrix
        transitions = np.zeros((model['n_states'], model['n_states']))
        for t in range(1, len(states)):
            transitions[states[t-1], states[t]] += 1
        
        # Normalize transitions
        for i in range(model['n_states']):
            if np.sum(transitions[i]) > 0:
                model['transition_matrix'][i] = transitions[i] / np.sum(transitions[i])
        
        # EM algorithm with fewer iterations and early stopping
        prev_ll = -np.inf
        
        for iteration in range(self.max_iter):
            # Pre-compute all emission log probabilities in parallel
            if self.n_jobs != 1:
                log_likelihoods = self._calculate_all_emissions_parallel(X_scaled, model)
            else:
                log_likelihoods = np.zeros((n_samples, model['n_states']))
                for i in range(model['n_states']):
                    for t in range(n_samples):
                        log_likelihoods[t, i] = self._emission_log_prob_vectorized(X_scaled[t], model, i)
            
            # Forward-backward algorithm
            log_alpha = self._forward_optimized(model, log_likelihoods)
            log_beta = self._backward_optimized(model, log_likelihoods)
            
            # State posterior probabilities (gamma)
            log_gamma = log_alpha + log_beta
            # Normalize using logsumexp for numerical stability
            normalizer = logsumexp(log_gamma, axis=1, keepdims=True)
            log_gamma = log_gamma - normalizer
            gamma = np.exp(log_gamma)
            
            # Calculate log-likelihood 
            current_ll = logsumexp(log_alpha[-1])
            
            # Check convergence - use relative improvement
            if prev_ll > -np.inf and (current_ll - prev_ll) / abs(prev_ll) < self.tol:
                break
                
            prev_ll = current_ll
            
            # M-step: Update parameters
            # Update initial probabilities
            model['initial_probs'] = gamma[0]
            
            # Update transition matrix efficiently
            xi_sum = np.zeros((model['n_states'], model['n_states']))
            
            # Pre-compute log transition matrix
            log_trans = np.log(model['transition_matrix'])
            
            # Calculate xi (transition probabilities) for all time steps
            for t in range(1, n_samples):
                # Calculate unnormalized xi for this time step
                for i in range(model['n_states']):
                    for j in range(model['n_states']):
                        xi_sum[i, j] += np.exp(log_alpha[t-1, i] + log_trans[i, j] + 
                                              log_likelihoods[t, j] + log_beta[t, j] - 
                                              logsumexp(log_alpha[-1]))
            
            # Normalize transition probabilities
            for i in range(model['n_states']):
                if np.sum(xi_sum[i]) > 0:
                    model['transition_matrix'][i] = xi_sum[i] / np.sum(xi_sum[i])
            
            # Update emission parameters more efficiently
            for i in range(model['n_states']):
                total_weight = np.sum(gamma[:, i])
                
                if total_weight > 0:
                    # Update location
                    model['location'][i] = np.sum(gamma[:, i].reshape(-1, 1) * X_scaled, axis=0) / total_weight
                    
                    # Update scale
                    diff = X_scaled - model['location'][i]
                    model['scale'][i] = np.sqrt(np.sum(gamma[:, i].reshape(-1, 1) * diff**2, axis=0) / total_weight)
                    model['scale'][i] = np.maximum(model['scale'][i], 1e-6)  # Prevent zero scale
                    
                    # Only update shape and df occasionally to save computation
                    if iteration % 2 == 0:
                        # Update shape (skewness)
                        if total_weight > 10:
                            # Standardize data
                            z = diff / model['scale'][i]
                            # Estimate skewness
                            model['shape'][i] = np.sum(gamma[:, i].reshape(-1, 1) * z**3, axis=0) / total_weight
                        
                        # Update degrees of freedom (tail heaviness)
                        if total_weight > 20 and iteration % 3 == 0:  # Even less frequent
                            # Standardize data
                            z = diff / model['scale'][i]
                            # Estimate kurtosis
                            kurt = np.sum(gamma[:, i].reshape(-1, 1) * z**4, axis=0) / total_weight - 3
                            # Convert to df parameter
                            model['df'][i] = np.maximum(4, np.minimum(30, 30 / (1 + np.abs(kurt))))
        
        # Viterbi algorithm for final state sequence
        model['states'] = self._predict(model, X_subset)
        model['log_likelihood'] = current_ll
        
        return model
    
    def _emission_log_prob_vectorized(self, x, model, state):
        """
        Vectorized calculation of log probability of observation x under given state 
        using Skew-t distribution approximation.
        
        Parameters:
        -----------
        x : numpy.ndarray
            Feature vector
        model : dict
            Model parameters
        state : int
            State index
            
        Returns:
        --------
        float
            Log probability
        """
        # Ensure x is 1D if needed
        if x.ndim == 0:
            x = np.array([x])
            
        # Standardize observation
        z = (x - model['location'][state]) / model['scale'][state]
        
        # Calculate normal log pdf (vectorized)
        log_pdf = np.sum(stats.norm.logpdf(z))
        
        # Add skewness effect (vectorized)
        # Apply skewness correction only where shape is significant
        shape_mask = np.abs(model['shape'][state]) > 1e-6
        if np.any(shape_mask):
            shape_correction = np.log(2) + stats.norm.logcdf(model['shape'][state][shape_mask] * z[shape_mask])
            log_pdf += np.sum(shape_correction)
        
        # Add heavy tail effect (vectorized)
        # Apply t-distribution correction only where df is low enough
        df_mask = model['df'][state] < 30
        if np.any(df_mask):
            # Vectorized calculation for t-distribution correction
            log_t = np.zeros_like(z[df_mask])
            log_norm = np.zeros_like(z[df_mask])
            
            for i, (z_val, df_val) in enumerate(zip(z[df_mask], model['df'][state][df_mask])):
                log_t[i] = stats.t.logpdf(z_val, df=df_val)
                log_norm[i] = stats.norm.logpdf(z_val)
            
            log_pdf += np.sum(log_t - log_norm)
        
        return log_pdf
    
    def _forward_optimized(self, model, log_likelihoods):
        """
        Optimized forward algorithm using log-space calculations.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        log_likelihoods : numpy.ndarray
            Pre-computed log emission probabilities
            
        Returns:
        --------
        numpy.ndarray
            Log forward probabilities
        """
        n_samples, n_states = log_likelihoods.shape
        log_alpha = np.zeros((n_samples, n_states))
        
        # Log of initial probabilities
        log_init = np.log(model['initial_probs'])
        
        # Log of transition matrix
        log_trans = np.log(model['transition_matrix'])
        
        # Initialize
        log_alpha[0] = log_init + log_likelihoods[0]
        
        # Forward recursion using vectorized operations and logsumexp
        for t in range(1, n_samples):
            for j in range(n_states):
                # Use logsumexp for numerical stability
                log_alpha[t, j] = logsumexp(log_alpha[t-1] + log_trans[:, j]) + log_likelihoods[t, j]
        
        return log_alpha
    
    def _backward_optimized(self, model, log_likelihoods):
        """
        Optimized backward algorithm using log-space calculations.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        log_likelihoods : numpy.ndarray
            Pre-computed log emission probabilities
            
        Returns:
        --------
        numpy.ndarray
            Log backward probabilities
        """
        n_samples, n_states = log_likelihoods.shape
        log_beta = np.zeros((n_samples, n_states))
        
        # Log of transition matrix
        log_trans = np.log(model['transition_matrix'])
        
        # Backward recursion using vectorized operations and logsumexp
        for t in range(n_samples-2, -1, -1):
            for i in range(n_states):
                log_terms = log_trans[i, :] + log_likelihoods[t+1, :] + log_beta[t+1, :]
                log_beta[t, i] = logsumexp(log_terms)
        
        return log_beta

### HMM with rolling window

In [11]:
class RollingWindowHMM(BaseHMMModel):
    """
    HMM with rolling windows to handle non-stationarity in time series data.
    Each window trains a separate HMM, and predictions are combined.
    """
    
    def __init__(self, n_states=3, window_size=730, step_size=21):
        """
        Initialize the RollingWindowHMM model.
        
        Parameters:
        -----------
        n_states : int
            Number of hidden states/regimes
        window_size : int
            Size of the rolling window (in samples)
        step_size : int
            Step size between consecutive windows (in samples)
        """
        super().__init__(n_states=n_states, model_name="RollingHMM")
        self.window_size = window_size
        self.step_size = step_size
    
    def _train_and_predict(self, X_subset, is_scaled):
        """
        Train RollingWindowHMM model and generate predictions.
        
        Parameters:
        -----------
        X_subset : numpy.ndarray
            Feature data subset
        is_scaled : bool
            Whether the data is already scaled
            
        Returns:
        --------
        tuple
            (trained_model, log_likelihood, regimes)
        """
        n_samples = X_subset.shape[0]
        n_features = X_subset.shape[1]
        
        # Check if we have enough data for at least one window
        if n_samples < self.window_size:
            raise ValueError(f"Not enough samples ({n_samples}) for window size {self.window_size}")
        
        # Create model dictionary to store all window models and parameters
        model = {
            'n_states': self.n_states,
            'window_size': self.window_size,
            'step_size': self.step_size,
            'n_features': n_features,
            'window_models': [],
            'window_indices': [],
            'window_likelihoods': [],
            'is_scaled': is_scaled,
            'scaler': None if is_scaled else RobustScaler(),
        }
        
        # Apply scaling if needed
        if model['scaler'] is not None:
            X_scaled = model['scaler'].fit_transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Calculate number of windows
        n_windows = max(1, (n_samples - self.window_size) // self.step_size + 1)
        
        # Train a model for each window
        for i in range(n_windows):
            window_start = i * self.step_size
            window_end = min(window_start + self.window_size, n_samples)
            
            # Extract window data
            window_data = X_scaled[window_start:window_end]
            
            # Create and train HMM for this window
            window_model = self._train_window_hmm(window_data, n_features)
            
            # Store window model and metadata
            model['window_models'].append(window_model)
            model['window_indices'].append((window_start, window_end))
            model['window_likelihoods'].append(window_model.score(window_data))
        
        # Combine predictions from all windows
        regimes = self._combine_window_predictions(model, X_scaled)
        
        # Calculate average log likelihood across all windows
        log_likelihood = np.mean(model['window_likelihoods']) if model['window_likelihoods'] else float('-inf')
        
        return model, log_likelihood, regimes
    
    def _predict(self, model, X_subset):
        """
        Make predictions with the trained RollingWindowHMM model.
        
        Parameters:
        -----------
        model : dict
            Trained model parameters
        X_subset : numpy.ndarray
            Feature data subset
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        # Apply scaling if needed
        if model.get('scaler') is not None:
            X_scaled = model['scaler'].transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Combine predictions from all windows
        return self._combine_window_predictions(model, X_scaled)
    
    def _train_window_hmm(self, window_data, n_features):
        """
        Train an HMM model for a specific window.
        
        Parameters:
        -----------
        window_data : numpy.ndarray
            Data for the current window
        n_features : int
            Number of features
            
        Returns:
        --------
        hmm.GaussianHMM
            Trained HMM model for this window
        """
        # Create appropriate HMM model based on dimensionality
        if n_features == 1:
            # For 1D data, reshape to 2D and use diagonal covariance
            window_model = hmm.GaussianHMM(
                n_components=self.n_states,
                covariance_type='diag',
                n_iter=100,
                random_state=42
            )
            # Ensure data is 2D
            if window_data.ndim == 1:
                window_data = window_data.reshape(-1, 1)
        else:
            # For multi-dimensional data, use full covariance
            window_model = hmm.GaussianHMM(
                n_components=self.n_states,
                covariance_type='full',
                n_iter=100,
                random_state=42
            )
        
        # Train the model with this window's data
        try:
            window_model.fit(window_data)
            return window_model
        except Exception as e:
            # If training fails, try with simplified model parameters
            print(f"Window model training failed, trying simplified model: {e}")
            window_model = hmm.GaussianHMM(
                n_components=self.n_states,
                covariance_type='diag',  # Always use diagonal for robustness
                n_iter=100,
                params='stmc',
                init_params='stmc',
                random_state=42
            )
            window_model.fit(window_data)
            return window_model
    
    def _combine_window_predictions(self, model, X_scaled):
        """
        Combine predictions from all window models.
        
        Parameters:
        -----------
        model : dict
            Model parameters with window models
        X_scaled : numpy.ndarray
            Scaled input data
            
        Returns:
        --------
        numpy.ndarray
            Combined regime predictions
        """
        n_samples = X_scaled.shape[0]
        regimes = np.zeros(n_samples, dtype=int)
        
        # If no window models, return all zeros
        if not model['window_models']:
            return regimes
        
        # Voting array to store predictions from each window
        votes = np.zeros((n_samples, self.n_states), dtype=int)
        
        # Get predictions from each window model
        for window_idx, (start, end) in enumerate(model['window_indices']):
            # Get the window model
            window_model = model['window_models'][window_idx]
            
            # Determine which samples this window can predict
            # We'll use predictions for samples that were in the window
            # and also extend to the next window's start if this is not the last window
            prediction_start = start
            
            if window_idx < len(model['window_indices']) - 1:
                prediction_end = min(model['window_indices'][window_idx + 1][0], n_samples)
            else:
                prediction_end = n_samples
            
            # Predict for this range
            try:
                window_data = X_scaled[start:end]
                window_states = window_model.predict(window_data)
                
                # Extend window predictions if needed
                if prediction_end > end:
                    # For samples beyond the window, use the last state
                    extension = np.full(prediction_end - end, window_states[-1])
                    window_predictions = np.concatenate([window_states, extension])
                else:
                    window_predictions = window_states[:prediction_end - start]
                
                # Register votes for the prediction range
                for i, state in enumerate(window_predictions):
                    if prediction_start + i < n_samples:
                        votes[prediction_start + i, state] += 1
            except Exception as e:
                print(f"Prediction failed for window {window_idx}: {e}")
        
        # Determine final states by majority voting
        for i in range(n_samples):
            if np.sum(votes[i]) > 0:
                regimes[i] = np.argmax(votes[i])
        
        return regimes

### Non Homogeneous HMM

In [12]:
class NonHomogeneousHMM(BaseHMMModel):
    """
    Non-homogeneous Hidden Markov Model where transition probabilities
    depend on external factors. Suitable for regime detection in financial data.
    """
    
    def __init__(self, n_states=3, external_factors=None):
        """
        Initialize the NonHomogeneousHMM model.
        
        Parameters:
        -----------
        n_states : int
            Number of hidden states/regimes
        external_factors : list of str, optional
            Names of features to use as external factors
            If None, the first feature will be used
        """
        super().__init__(n_states=n_states, model_name="NonHomoHMM")
        self.external_factors = external_factors
    
    def train_all_models(self, X_train, X_train_scaled, features, external_factors_list=None):
        """
        Train models on all feature combinations with both scaled and unscaled data.
        Overrides the base class method to handle external factors.
        
        Parameters:
        -----------
        X_train : pandas.DataFrame
            Raw training data
        X_train_scaled : pandas.DataFrame
            Scaled training data (e.g., using RobustScaler)
        features : list
            List of feature combinations to use, each item is a list of feature names
        external_factors_list : list, optional
            List of features to use as external factors
            If None, uses the first feature of each combination
            
        Returns:
        --------
        dict
            Updated models_info dictionary with all trained models
        """
        # If no external factors list provided, use default approach
        if external_factors_list is None:
            all_columns = X_train.columns.tolist()
            external_factors_list = all_columns[:1]  # Use first column by default
        
        # Train models for each feature combination, external factor, and scaling option
        for external_factor in external_factors_list:
            print(f"\nTraining with external factor: {external_factor}")
            
            for is_scaled in [False, True]:
                # Select appropriate data
                X = X_train_scaled if is_scaled else X_train
                scaling_name = "robust" if is_scaled else "none"
                
                print(f"  Using {scaling_name} scaling")
                
                # Iterate through each feature combination
                for feature_list in features:
                    # Skip if external factor is in feature list (to avoid using same feature twice)
                    if external_factor in feature_list:
                        continue
                    
                    feature_name = "+".join(feature_list)
                    
                    try:
                        # Extract data for features and external factor
                        X_subset = X[feature_list].values
                        external_data = X[external_factor].values.reshape(-1, 1)
                        
                        # Train the model
                        model, log_likelihood, regimes = self._train_and_predict(
                            X_subset, external_data, is_scaled, external_factor
                        )
                        
                        # Calculate silhouette score (handle single feature case)
                        silhouette = self._calculate_silhouette(X_subset, regimes)
                        
                        # Generate unique model ID
                        model_id = f"{self.model_name}_{external_factor}_{scaling_name}_{feature_name}_{self.model_id_counter}"
                        self.model_id_counter += 1
                        
                        # Store model information
                        self.models_info[model_id] = {
                            'model': model,
                            'model_type': self.model_name,
                            'external_factor': external_factor,
                            'scaling': scaling_name,
                            'features': feature_list,
                            'silhouette_score': silhouette,
                            'log_likelihood': log_likelihood,
                            'regimes': regimes
                        }
                        
                    except Exception as e:
                        print(f"Error training model with features {feature_list}, external factor {external_factor}, scaling {scaling_name}: {str(e)}")
        
        return self.models_info
    
    def _train_and_predict(self, X_subset, external_data, is_scaled, external_factor_name):
        """
        Train NonHomogeneousHMM model and generate predictions.
        
        Parameters:
        -----------
        X_subset : numpy.ndarray
            Feature data subset
        external_data : numpy.ndarray
            External factor data
        is_scaled : bool
            Whether the data is already scaled
        external_factor_name : str
            Name of the external factor
            
        Returns:
        --------
        tuple
            (trained_model, log_likelihood, regimes)
        """
        # Initialize model parameters
        model = self._create_model(X_subset.shape[1], external_data.shape[1], is_scaled, external_factor_name)
        
        # Fit the model
        model = self._fit_model(model, X_subset, external_data)
        
        # Generate predictions
        regimes = self._viterbi_predict(model, X_subset, external_data)
        
        # Calculate log likelihood
        log_likelihood = self._forward_log_likelihood(model, X_subset, external_data)
        
        return model, log_likelihood, regimes
    
    def _predict(self, model, X_subset):
        """
        Make predictions with the trained NonHomogeneousHMM model.
        
        Parameters:
        -----------
        model : dict
            Trained model parameters
        X_subset : numpy.ndarray
            Feature data subset
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        # Get the external factor data from the features
        external_factor_name = model['external_factor']
        
        # Check if X_subset is a DataFrame or a NumPy array
        if hasattr(X_subset, 'columns'):
            # X_subset is a DataFrame
            if external_factor_name in X_subset.columns:
                external_data = X_subset[external_factor_name].values.reshape(-1, 1)
                # Remove external factor from features if it's there
                X_features = X_subset.drop(columns=[external_factor_name]).values
            else:
                # External factor not found, use a default approach
                print(f"Warning: External factor {external_factor_name} not found. Using first column.")
                external_data = X_subset.iloc[:, 0].values.reshape(-1, 1)
                X_features = X_subset.iloc[:, 1:].values
        else:
            # X_subset is a NumPy array, assume external factor is the first column
            print("Warning: Input is NumPy array. Assuming first column is external factor.")
            external_data = X_subset[:, 0].reshape(-1, 1)
            X_features = X_subset[:, 1:]
        
        # Apply scaling if needed
        if model.get('scaler') is not None:
            X_scaled = model['scaler'].transform(X_features)
            external_scaled = model['external_scaler'].transform(external_data)
        else:
            X_scaled = X_features
            external_scaled = external_data
        
        # Get predictions using Viterbi algorithm
        return self._viterbi_predict(model, X_scaled, external_scaled)
    
    def _create_model(self, n_features, n_external, is_scaled, external_factor_name):
        """
        Create a new NonHomogeneousHMM model with initialized parameters.
        
        Parameters:
        -----------
        n_features : int
            Number of features in the data
        n_external : int
            Number of external factors
        is_scaled : bool
            Whether the data is already scaled
        external_factor_name : str
            Name of the external factor
            
        Returns:
        --------
        dict
            Model parameters dictionary
        """
        # Initialize model parameters
        model = {
            'n_states': self.n_states,
            'n_features': n_features,
            'n_external': n_external,
            'external_factor': external_factor_name,
            'initial_probs': np.ones(self.n_states) / self.n_states,
            
            # Parameters for transition probability calculation
            # For each transition (i,j), we have weights for [bias, external_factor_1, ..., external_factor_n]
            'transition_params': np.zeros((self.n_states, self.n_states, n_external + 1)),
            
            # Emission parameters
            'means': np.zeros((self.n_states, n_features)),
            'covars': np.ones((self.n_states, n_features, n_features)),
            
            # Scalers
            'scaler': None if is_scaled else RobustScaler(),
            'external_scaler': None if is_scaled else RobustScaler(),
            
            # For tracking convergence
            'log_likelihood_history': []
        }
        
        # Initialize transition parameters with small random values
        model['transition_params'] = np.random.randn(self.n_states, self.n_states, n_external + 1) * 0.01
        
        # Set bias term to favor self-transitions slightly (diagonal)
        for i in range(self.n_states):
            model['transition_params'][i, i, 0] = 0.1
        
        return model
    
    def _fit_model(self, model, X_subset, external_data, max_iter=100, tol=1e-6):
        """
        Fit the NonHomogeneousHMM model using EM algorithm.
        
        Parameters:
        -----------
        model : dict
            Model parameters dictionary
        X_subset : numpy.ndarray
            Feature data subset
        external_data : numpy.ndarray
            External factor data
        max_iter : int
            Maximum number of iterations
        tol : float
            Convergence tolerance
            
        Returns:
        --------
        dict
            Updated model parameters
        """
        n_samples = X_subset.shape[0]
        
        # Apply scaling if needed
        if model['scaler'] is not None:
            X_scaled = model['scaler'].fit_transform(X_subset)
            external_scaled = model['external_scaler'].fit_transform(external_data)
        else:
            X_scaled = X_subset
            external_scaled = external_data
        
        # Initialize parameters using k-means
        kmeans = KMeans(n_clusters=model['n_states'], random_state=42)
        states = kmeans.fit_predict(X_scaled)
        
        # Initialize emission parameters based on clusters
        for i in range(model['n_states']):
            mask = (states == i)
            if np.sum(mask) > 0:
                # Initialize means
                model['means'][i] = np.mean(X_scaled[mask], axis=0)
                
                # Initialize covariances with regularization
                if X_scaled.shape[1] > 1:
                    # Multi-dimensional case
                    model['covars'][i] = np.cov(X_scaled[mask], rowvar=False) + 1e-6 * np.eye(X_scaled.shape[1])
                else:
                    # One-dimensional case
                    model['covars'][i] = np.array([[np.var(X_scaled[mask]) + 1e-6]])
        
        # Main EM loop
        prev_ll = -np.inf
        
        for iteration in range(max_iter):
            # E-step: Forward-backward
            log_alpha = self._forward(model, X_scaled, external_scaled)
            log_beta = self._backward(model, X_scaled, external_scaled)
            
            # Calculate posteriors (gamma)
            log_gamma = log_alpha + log_beta
            # Normalize
            log_norm = logsumexp(log_gamma, axis=1, keepdims=True)
            log_gamma = log_gamma - log_norm
            gamma = np.exp(log_gamma)
            
            # Calculate log-likelihood
            current_ll = logsumexp(log_alpha[-1, :])
            model['log_likelihood_history'].append(current_ll)
            
            # Check convergence
            if np.abs(current_ll - prev_ll) < tol:
                break
                
            prev_ll = current_ll
            
            # M-step: Update parameters
            # Update initial probabilities
            model['initial_probs'] = gamma[0, :]
            
            # Update emission parameters
            for i in range(model['n_states']):
                # Update means
                weighted_sum = np.sum(gamma[:, i][:, np.newaxis] * X_scaled, axis=0)
                model['means'][i] = weighted_sum / (np.sum(gamma[:, i]) + 1e-10)
                
                # Update covariances
                diff = X_scaled - model['means'][i]
                
                if X_scaled.shape[1] > 1:
                    # Multi-dimensional case
                    weighted_outer = np.zeros((X_scaled.shape[1], X_scaled.shape[1]))
                    for t in range(n_samples):
                        weighted_outer += gamma[t, i] * np.outer(diff[t], diff[t])
                    model['covars'][i] = weighted_outer / (np.sum(gamma[:, i]) + 1e-10)
                    # Add regularization
                    model['covars'][i] += 1e-6 * np.eye(X_scaled.shape[1])
                else:
                    # One-dimensional case
                    model['covars'][i] = np.array([[
                        np.sum(gamma[:, i] * diff.flatten()**2) / (np.sum(gamma[:, i]) + 1e-10) + 1e-6
                    ]])
            
            # Update transition parameters using gradient descent
            self._update_transition_params(model, X_scaled, external_scaled, gamma)
        
        return model
    
    def _update_transition_params(self, model, X, external_data, gamma, learning_rate=0.01):
        """
        Update transition parameters using gradient descent.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        X : numpy.ndarray
            Feature data
        external_data : numpy.ndarray
            External factor data
        gamma : numpy.ndarray
            State posteriors
        learning_rate : float
            Learning rate for gradient descent
            
        Returns:
        --------
        None (updates model in-place)
        """
        n_samples = X.shape[0]
        
        # Calculate xi (joint probability of being in state i at t and state j at t+1)
        xi = np.zeros((n_samples - 1, model['n_states'], model['n_states']))
        
        for t in range(n_samples - 1):
            # Get transition matrix for this time step
            trans_mat = self._transition_matrix(model, external_data[t])
            
            # Calculate emission probabilities for t+1
            emit_probs = np.zeros(model['n_states'])
            for j in range(model['n_states']):
                emit_probs[j] = self._emission_prob(model, X[t+1], j)
            
            # Calculate unnormalized xi
            for i in range(model['n_states']):
                for j in range(model['n_states']):
                    xi[t, i, j] = gamma[t, i] * trans_mat[i, j] * emit_probs[j]
            
            # Normalize
            if np.sum(xi[t]) > 0:
                xi[t] /= np.sum(xi[t])
        
        # Update transition parameters
        for i in range(model['n_states']):
            for j in range(model['n_states']):
                # Initialize gradient
                gradient = np.zeros(model['n_external'] + 1)
                
                # Calculate gradient
                for t in range(n_samples - 1):
                    # Prepare input vector [1, external_factor_1, ..., external_factor_n]
                    x_t = np.concatenate([np.ones(1), external_data[t].flatten()])
                    
                    # Get transition matrix for this time step
                    trans_mat = self._transition_matrix(model, external_data[t])
                    
                    # Calculate error (difference between expected and actual transition)
                    error = xi[t, i, j] - gamma[t, i] * trans_mat[i, j]
                    
                    # Update gradient
                    gradient += error * x_t
                
                # Apply gradient update
                model['transition_params'][i, j] += learning_rate * gradient
    
    def _transition_matrix(self, model, external_factors):
        """
        Calculate transition probability matrix based on external factors.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        external_factors : numpy.ndarray
            External factors for the current time step
            
        Returns:
        --------
        numpy.ndarray
            Transition probability matrix
        """
        # Prepare input vector [1, external_factor_1, ..., external_factor_n]
        x = np.concatenate([np.ones(1), external_factors.flatten()])
        
        # Calculate logits
        logits = np.zeros((model['n_states'], model['n_states']))
        for i in range(model['n_states']):
            for j in range(model['n_states']):
                logits[i, j] = np.dot(model['transition_params'][i, j], x)
        
        # Convert to probabilities using softmax
        trans_probs = np.zeros_like(logits)
        for i in range(model['n_states']):
            # Use logsumexp for numerical stability
            log_norm = logsumexp(logits[i])
            for j in range(model['n_states']):
                trans_probs[i, j] = np.exp(logits[i, j] - log_norm)
        
        return trans_probs
    
    def _emission_prob(self, model, x, state):
        """
        Calculate emission probability for observation x under given state.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        x : numpy.ndarray
            Feature vector
        state : int
            State index
            
        Returns:
        --------
        float
            Emission probability
        """
        # Ensure x is 1D if needed
        if x.ndim == 0:
            x = np.array([x])
        
        # Handle one-dimensional case specially
        if model['n_features'] == 1:
            return stats.norm.pdf(
                x.flatten(),
                loc=model['means'][state].flatten(),
                scale=np.sqrt(model['covars'][state][0, 0])
            )[0]
        else:
            return stats.multivariate_normal.pdf(
                x,
                mean=model['means'][state],
                cov=model['covars'][state]
            )
    
    def _emission_log_prob(self, model, x, state):
        """
        Calculate log emission probability for observation x under given state.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        x : numpy.ndarray
            Feature vector
        state : int
            State index
            
        Returns:
        --------
        float
            Log emission probability
        """
        # Ensure x is 1D if needed
        if x.ndim == 0:
            x = np.array([x])
        
        # Handle one-dimensional case specially
        if model['n_features'] == 1:
            return stats.norm.logpdf(
                x.flatten(),
                loc=model['means'][state].flatten(),
                scale=np.sqrt(model['covars'][state][0, 0])
            )[0]
        else:
            return stats.multivariate_normal.logpdf(
                x,
                mean=model['means'][state],
                cov=model['covars'][state]
            )
    
    def _forward(self, model, X, external_data):
        """
        Forward algorithm for NonHomogeneousHMM.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        X : numpy.ndarray
            Feature data
        external_data : numpy.ndarray
            External factor data
            
        Returns:
        --------
        numpy.ndarray
            Log forward probabilities
        """
        n_samples = X.shape[0]
        n_states = model['n_states']
        log_alpha = np.zeros((n_samples, n_states))
        
        # Initialize
        for i in range(n_states):
            log_alpha[0, i] = np.log(model['initial_probs'][i]) + self._emission_log_prob(model, X[0], i)
        
        # Forward recursion
        for t in range(1, n_samples):
            # Get transition matrix for this time step
            trans_mat = self._transition_matrix(model, external_data[t-1])
            
            for j in range(n_states):
                # Calculate log sum of exp
                log_sum = np.zeros(n_states)
                for i in range(n_states):
                    log_sum[i] = log_alpha[t-1, i] + np.log(trans_mat[i, j])
                
                # Use logsumexp for numerical stability
                log_alpha[t, j] = logsumexp(log_sum) + self._emission_log_prob(model, X[t], j)
        
        return log_alpha
    
    def _backward(self, model, X, external_data):
        """
        Backward algorithm for NonHomogeneousHMM.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        X : numpy.ndarray
            Feature data
        external_data : numpy.ndarray
            External factor data
            
        Returns:
        --------
        numpy.ndarray
            Log backward probabilities
        """
        n_samples = X.shape[0]
        n_states = model['n_states']
        log_beta = np.zeros((n_samples, n_states))
        
        # Initialize (all log(1) = 0)
        
        # Backward recursion
        for t in range(n_samples-2, -1, -1):
            # Get transition matrix for this time step
            trans_mat = self._transition_matrix(model, external_data[t])
            
            for i in range(n_states):
                # Calculate log sum of exp
                log_sum = np.zeros(n_states)
                for j in range(n_states):
                    log_sum[j] = np.log(trans_mat[i, j]) + \
                               self._emission_log_prob(model, X[t+1], j) + \
                               log_beta[t+1, j]
                
                # Use logsumexp for numerical stability
                log_beta[t, i] = logsumexp(log_sum)
        
        return log_beta
    
    def _forward_log_likelihood(self, model, X, external_data):
        """
        Calculate log likelihood using forward algorithm.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        X : numpy.ndarray
            Feature data
        external_data : numpy.ndarray
            External factor data
            
        Returns:
        --------
        float
            Log likelihood
        """
        log_alpha = self._forward(model, X, external_data)
        return logsumexp(log_alpha[-1, :])
    
    def _viterbi_predict(self, model, X, external_data):
        """
        Viterbi algorithm for finding most likely state sequence.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        X : numpy.ndarray
            Feature data
        external_data : numpy.ndarray
            External factor data
            
        Returns:
        --------
        numpy.ndarray
            Most likely state sequence
        """
        n_samples = X.shape[0]
        n_states = model['n_states']
        
        # Initialize
        delta = np.zeros((n_samples, n_states))
        psi = np.zeros((n_samples, n_states), dtype=int)
        
        # Initialization
        for i in range(n_states):
            delta[0, i] = np.log(model['initial_probs'][i]) + self._emission_log_prob(model, X[0], i)
        
        # Recursion
        for t in range(1, n_samples):
            # Get transition matrix for this time step
            trans_mat = self._transition_matrix(model, external_data[t-1])
            
            for j in range(n_states):
                # Find the most likely previous state
                probs = delta[t-1, :] + np.log(trans_mat[:, j])
                psi[t, j] = np.argmax(probs)
                delta[t, j] = probs[psi[t, j]] + self._emission_log_prob(model, X[t], j)
        
        # Backtracking
        q = np.zeros(n_samples, dtype=int)
        q[-1] = np.argmax(delta[-1, :])
        
        for t in range(n_samples-2, -1, -1):
            q[t] = psi[t+1, q[t+1]]
        
        return q

### HMM with GMM

In [13]:
class HMMWithGMM(BaseHMMModel):
    """
    Hidden Markov Model with Gaussian Mixture Models for emission distributions.
    Better handles multimodal and complex distributions within each state.
    """
    
    def __init__(self, n_states=3, n_mixtures=3):
        """
        Initialize the HMMWithGMM model.
        
        Parameters:
        -----------
        n_states : int
            Number of hidden states/regimes
        n_mixtures : int
            Number of Gaussian mixtures per state
        """
        super().__init__(n_states=n_states, model_name="GMMHMM")
        self.n_mixtures = n_mixtures
    
    def _train_and_predict(self, X_subset, is_scaled):
        """
        Train HMMWithGMM model and generate predictions.
        
        Parameters:
        -----------
        X_subset : numpy.ndarray
            Feature data subset
        is_scaled : bool
            Whether the data is already scaled
            
        Returns:
        --------
        tuple
            (trained_model, log_likelihood, regimes)
        """
        n_samples, n_features = X_subset.shape
        
        # Initialize model dictionary
        model = {
            'n_states': self.n_states,
            'n_mixtures': self.n_mixtures,
            'n_features': n_features,
            'hmm_model': None,
            'gmm_models': [None] * self.n_states,
            'scaler': None if is_scaled else RobustScaler(),
            'log_likelihood': 0.0,
            'is_scaled': is_scaled
        }
        
        # Apply scaling if needed
        if model['scaler'] is not None:
            X_scaled = model['scaler'].fit_transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Two-step approach:
        # 1. Train a standard HMM to get initial state assignments
        # 2. Train a GMM for each state based on those assignments
        
        # Step 1: Train a standard HMM
        try:
            # Determine covariance type based on dimensionality and sample size
            if n_features == 1:
                cov_type = 'diag'  # Diagonal for 1D data
                # Ensure data is 2D for hmmlearn
                X_hmm = X_scaled.reshape(-1, 1)
            else:
                # Use 'full' if enough samples, otherwise 'diag' or 'tied'
                if n_samples >= n_features * 10:
                    cov_type = 'full'
                else:
                    cov_type = 'diag'
                X_hmm = X_scaled
            
            # Create and train HMM
            hmm_model = hmm.GaussianHMM(
                n_components=self.n_states,
                covariance_type=cov_type,
                n_iter=100,
                random_state=42
            )
            hmm_model.fit(X_hmm)
            
            # Get state predictions
            state_seq = hmm_model.predict(X_hmm)
            
            # Store HMM model
            model['hmm_model'] = hmm_model
            model['log_likelihood'] = hmm_model.score(X_hmm)
            
        except Exception as e:
            print(f"HMM training failed: {e}")
            print("Falling back to K-means clustering for initial state assignment")
            
            # Fallback to K-means clustering
            kmeans = KMeans(n_clusters=self.n_states, random_state=42)
            state_seq = kmeans.fit_predict(X_scaled)
            
            # Create a simple HMM with means based on clusters
            means = np.zeros((self.n_states, n_features))
            covars = np.zeros((self.n_states, n_features, n_features))
            
            for i in range(self.n_states):
                mask = (state_seq == i)
                if np.sum(mask) > 0:
                    means[i] = np.mean(X_scaled[mask], axis=0)
                    if n_features > 1:
                        covars[i] = np.cov(X_scaled[mask], rowvar=False) + 1e-6 * np.eye(n_features)
                    else:
                        covars[i] = np.array([[np.var(X_scaled[mask]) + 1e-6]])
            
            # Calculate a simple log likelihood
            log_like = 0.0
            for i in range(n_samples):
                state = state_seq[i]
                if n_features > 1:
                    log_like += stats.multivariate_normal.logpdf(
                        X_scaled[i], mean=means[state], cov=covars[state]
                    )
                else:
                    log_like += stats.norm.logpdf(
                        X_scaled[i], loc=means[state], scale=np.sqrt(covars[state][0, 0])
                    )
            
            model['log_likelihood'] = log_like
        
        # Step 2: Train a GMM for each state
        for state in range(self.n_states):
            # Get samples assigned to this state
            mask = (state_seq == state)
            state_samples = X_scaled[mask]
            
            # Skip if not enough samples
            if len(state_samples) <= self.n_mixtures:
                print(f"Warning: State {state} has only {len(state_samples)} samples, need at least {self.n_mixtures+1}")
                continue
            
            try:
                # Determine actual number of mixtures (might need to reduce if few samples)
                actual_n_mixtures = min(self.n_mixtures, len(state_samples) // 2)
                
                # Determine covariance type based on dimensionality and sample size
                if len(state_samples) >= n_features * 10 * actual_n_mixtures:
                    gmm_cov_type = 'full'
                else:
                    gmm_cov_type = 'diag'
                
                # Create and fit GMM
                gmm = GaussianMixture(
                    n_components=actual_n_mixtures,
                    covariance_type=gmm_cov_type,
                    random_state=42
                )
                gmm.fit(state_samples)
                
                # Store GMM model
                model['gmm_models'][state] = gmm
                
            except Exception as e:
                print(f"GMM training failed for state {state}: {e}")
                # Leave as None, will handle in prediction
        
        # Use HMM for overall state sequence prediction
        regimes = state_seq
        
        return model, model['log_likelihood'], regimes
    
    def _predict(self, model, X_subset):
        """
        Make predictions with the trained HMMWithGMM model.
        
        Parameters:
        -----------
        model : dict
            Trained model parameters
        X_subset : numpy.ndarray
            Feature data subset
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        # Apply scaling if needed
        if model.get('scaler') is not None:
            X_scaled = model['scaler'].transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Make prediction using the HMM model
        if model['hmm_model'] is not None:
            # Ensure data is 2D for hmmlearn
            if X_scaled.ndim == 1:
                X_hmm = X_scaled.reshape(-1, 1)
            else:
                X_hmm = X_scaled
            
            # Use HMM for state prediction
            return model['hmm_model'].predict(X_hmm)
        else:
            # Fallback to manual prediction using emission probabilities
            return self._manual_predict(model, X_scaled)
    
    def _manual_predict(self, model, X_scaled):
        """
        Manual prediction when HMM model is not available.
        Uses Viterbi algorithm with emission probabilities from GMMs.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        X_scaled : numpy.ndarray
            Scaled input data
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        n_samples = X_scaled.shape[0]
        n_states = model['n_states']
        
        # Initialize Viterbi variables
        delta = np.zeros((n_samples, n_states))
        psi = np.zeros((n_samples, n_states), dtype=int)
        
        # Get initial state probabilities (assume uniform if not available)
        if model['hmm_model'] is not None:
            initial_probs = model['hmm_model'].startprob_
        else:
            initial_probs = np.ones(n_states) / n_states
        
        # Get transition matrix (assume uniform if not available)
        if model['hmm_model'] is not None:
            trans_mat = model['hmm_model'].transmat_
        else:
            trans_mat = np.ones((n_states, n_states)) / n_states
        
        # Initialization
        for i in range(n_states):
            delta[0, i] = np.log(initial_probs[i]) + self._emission_log_prob(model, X_scaled[0], i)
        
        # Recursion
        for t in range(1, n_samples):
            for j in range(n_states):
                # Find the most likely previous state
                probs = delta[t-1, :] + np.log(trans_mat[:, j])
                psi[t, j] = np.argmax(probs)
                delta[t, j] = probs[psi[t, j]] + self._emission_log_prob(model, X_scaled[t], j)
        
        # Backtracking
        q = np.zeros(n_samples, dtype=int)
        q[-1] = np.argmax(delta[-1, :])
        
        for t in range(n_samples-2, -1, -1):
            q[t] = psi[t+1, q[t+1]]
        
        return q
    
    def _emission_log_prob(self, model, x, state):
        """
        Calculate log probability of observation x under given state.
        Uses GMM if available, otherwise falls back to normal distribution.
        
        Parameters:
        -----------
        model : dict
            Model parameters
        x : numpy.ndarray
            Feature vector
        state : int
            State index
            
        Returns:
        --------
        float
            Log probability
        """
        # Ensure x is 2D for sklearn
        if x.ndim == 1:
            x_2d = x.reshape(1, -1)
        else:
            x_2d = x
        
        # If GMM available for this state, use it
        if model['gmm_models'][state] is not None:
            try:
                return model['gmm_models'][state].score_samples(x_2d)[0]
            except Exception as e:
                print(f"GMM score_samples failed: {e}")
                # Fall through to fallback
        
        # Fallback: use normal distribution from HMM model if available
        if model['hmm_model'] is not None:
            try:
                # Extract means and covariances from HMM model
                mean = model['hmm_model'].means_[state]
                
                # Handle different covariance types
                if model['hmm_model'].covariance_type == 'full':
                    cov = model['hmm_model'].covars_[state]
                elif model['hmm_model'].covariance_type == 'diag':
                    cov = np.diag(model['hmm_model'].covars_[state])
                elif model['hmm_model'].covariance_type == 'tied':
                    cov = model['hmm_model'].covars_
                else:  # 'spherical'
                    cov = model['hmm_model'].covars_[state] * np.eye(model['n_features'])
                
                # Calculate log pdf
                if model['n_features'] > 1:
                    return stats.multivariate_normal.logpdf(x, mean=mean, cov=cov)
                else:
                    return stats.norm.logpdf(x, loc=mean, scale=np.sqrt(cov[0, 0]))
            except Exception as e:
                print(f"HMM emission probability calculation failed: {e}")
                # Fall through to final fallback
        
        # Final fallback: return a small constant log probability
        return -10.0  # Small log probability as fallback
    
    def score_samples(self, model, X_subset):
        """
        Score samples using the model.
        Returns log-likelihood for each sample.
        
        Parameters:
        -----------
        model : dict
            Trained model
        X_subset : numpy.ndarray
            Input data
            
        Returns:
        --------
        numpy.ndarray
            Log-likelihood scores for each sample
        """
        # Apply scaling if needed
        if model.get('scaler') is not None:
            X_scaled = model['scaler'].transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Get state assignments
        states = self._predict(model, X_scaled)
        
        # Calculate log-likelihood for each sample using the appropriate GMM
        n_samples = X_scaled.shape[0]
        scores = np.zeros(n_samples)
        
        for i in range(n_samples):
            scores[i] = self._emission_log_prob(model, X_scaled[i], states[i])
        
        return scores
    
    def get_state_distributions(self, model):
        """
        Get the distribution parameters for each state.
        
        Parameters:
        -----------
        model : dict
            Trained model
            
        Returns:
        --------
        dict
            Distribution parameters for each state
        """
        state_dists = {}
        
        for state in range(model['n_states']):
            if model['gmm_models'][state] is not None:
                gmm = model['gmm_models'][state]
                
                state_dists[state] = {
                    'means': gmm.means_,
                    'covariances': gmm.covariances_,
                    'weights': gmm.weights_
                }
            elif model['hmm_model'] is not None:
                # Extract from HMM model
                state_dists[state] = {
                    'means': [model['hmm_model'].means_[state]],
                    'weights': [1.0]
                }
                
                # Handle different covariance types
                if model['hmm_model'].covariance_type == 'full':
                    state_dists[state]['covariances'] = [model['hmm_model'].covars_[state]]
                elif model['hmm_model'].covariance_type == 'diag':
                    state_dists[state]['covariances'] = [np.diag(model['hmm_model'].covars_[state])]
                elif model['hmm_model'].covariance_type == 'tied':
                    state_dists[state]['covariances'] = [model['hmm_model'].covars_]
                else:  # 'spherical'
                    state_dists[state]['covariances'] = [
                        model['hmm_model'].covars_[state] * np.eye(model['n_features'])
                    ]
            else:
                # No distribution information available
                state_dists[state] = {
                    'means': None,
                    'covariances': None,
                    'weights': None
                }
        
        return state_dists

### HMM with Factor analysis

In [14]:
class HierarchicalHMM(BaseHMMModel):
    """
    Hierarchical Hidden Markov Model using factor analysis for dimensionality reduction.
    First extracts latent factors, then applies HMM on the factor space.
    """
    
    def __init__(self, n_states=3, n_factors=None):
        """
        Initialize the HierarchicalHMM model.
        
        Parameters:
        -----------
        n_states : int
            Number of hidden states/regimes
        n_factors : int or None
            Number of latent factors to extract
            If None, will be determined automatically
        """
        super().__init__(n_states=n_states, model_name="HierarchicalHMM")
        self.n_factors = n_factors
    
    def train_all_models(self, X_train, X_train_scaled, features):
        """
        Train models on all feature combinations with both scaled and unscaled data.
        Overrides the base class method to handle different factor counts.
        
        Parameters:
        -----------
        X_train : pandas.DataFrame
            Raw training data
        X_train_scaled : pandas.DataFrame
            Scaled training data (e.g., using RobustScaler)
        features : list
            List of feature combinations to use, each item is a list of feature names
            
        Returns:
        --------
        dict
            Updated models_info dictionary with all trained models
        """
        # Train models for each feature combination and scaling option
        for is_scaled in [False, True]:
            # Select appropriate data
            X = X_train_scaled if is_scaled else X_train
            scaling_name = "robust" if is_scaled else "none"
            
            print(f"\nTraining {self.model_name} models with {scaling_name} scaling")
            
            # Only use feature combinations with at least 2 features
            valid_features = [feat_list for feat_list in features if len(feat_list) >= 2]
            
            if not valid_features:
                print(f"Warning: No feature combinations with 2+ features found. Hierarchical HMM requires at least 2 features.")
                continue
            
            # Iterate through each feature combination
            for feature_list in valid_features:
                feature_name = "+".join(feature_list)
                n_features = len(feature_list)
                
                # Determine possible factor counts
                max_factors = min(n_features - 1, 5)  # At most 5 factors or n_features-1
                
                # Try different factor counts
                for n_factors in range(1, max_factors + 1):
                    try:
                        # Extract data for this feature combination
                        X_subset = X[feature_list].values
                        
                        # Train the model with this factor count
                        model, log_likelihood, regimes = self._train_and_predict(X_subset, is_scaled, n_factors)
                        
                        # Calculate silhouette score
                        silhouette = self._calculate_silhouette(X_subset, regimes)
                        
                        # Generate unique model ID
                        model_id = f"{self.model_name}_{n_factors}factors_{scaling_name}_{feature_name}_{self.model_id_counter}"
                        self.model_id_counter += 1
                        
                        # Store model information
                        self.models_info[model_id] = {
                            'model': model,
                            'model_type': self.model_name,
                            'scaling': scaling_name,
                            'features': feature_list,
                            'n_factors': n_factors,
                            'silhouette_score': silhouette,
                            'log_likelihood': log_likelihood,
                            'regimes': regimes
                        }
                        
                    except Exception as e:
                        print(f"Error training model with features {feature_list}, {n_factors} factors, scaling {scaling_name}: {str(e)}")
        
        return self.models_info
    
    def _train_and_predict(self, X_subset, is_scaled, n_factors):
        """
        Train HierarchicalHMM model and generate predictions.
        
        Parameters:
        -----------
        X_subset : numpy.ndarray
            Feature data subset
        is_scaled : bool
            Whether the data is already scaled
        n_factors : int
            Number of latent factors to extract
            
        Returns:
        --------
        tuple
            (trained_model, log_likelihood, regimes)
        """
        n_samples, n_features = X_subset.shape
        
        # Initialize model dictionary
        model = {
            'n_states': self.n_states,
            'n_factors': n_factors,
            'n_features': n_features,
            'hmm_model': None,
            'factor_model': None,
            'scaler': None if is_scaled else RobustScaler(),
            'log_likelihood': 0.0,
            'is_scaled': is_scaled
        }
        
        # Apply scaling if needed
        if model['scaler'] is not None:
            X_scaled = model['scaler'].fit_transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Step 1: Extract latent factors
        try:
            # Try Factor Analysis first
            fa = FactorAnalysis(
                n_components=n_factors,
                random_state=42
            )
            factor_scores = fa.fit_transform(X_scaled)
            model['factor_model'] = fa
            model['factor_method'] = 'FactorAnalysis'
            
            # Calculate the explained variance
            explained_variance = fa.get_covariance().diagonal().sum()
            total_variance = np.var(X_scaled, axis=0).sum()
            model['explained_variance_ratio'] = explained_variance / total_variance
            
        except Exception as e:
            print(f"Factor Analysis failed: {e}")
            print("Falling back to PCA")
            
            # Fallback to PCA
            pca = PCA(
                n_components=n_factors,
                random_state=42
            )
            factor_scores = pca.fit_transform(X_scaled)
            model['factor_model'] = pca
            model['factor_method'] = 'PCA'
            model['explained_variance_ratio'] = sum(pca.explained_variance_ratio_)
        
        # Step 2: Train HMM on factor scores
        try:
            # Create and train HMM on factor scores
            hmm_model = hmm.GaussianHMM(
                n_components=self.n_states,
                covariance_type='full',  # Use full for factor scores
                n_iter=100,
                random_state=42
            )
            hmm_model.fit(factor_scores)
            
            # Get state predictions
            regimes = hmm_model.predict(factor_scores)
            
            # Store HMM model
            model['hmm_model'] = hmm_model
            model['log_likelihood'] = hmm_model.score(factor_scores)
            
        except Exception as e:
            print(f"HMM training failed: {e}")
            print("Falling back to simplified HMM")
            
            # Try with simplified HMM (diagonal covariance)
            try:
                hmm_model = hmm.GaussianHMM(
                    n_components=self.n_states,
                    covariance_type='diag',  # Use diagonal instead
                    n_iter=100,
                    random_state=42
                )
                hmm_model.fit(factor_scores)
                
                # Get state predictions
                regimes = hmm_model.predict(factor_scores)
                
                # Store HMM model
                model['hmm_model'] = hmm_model
                model['log_likelihood'] = hmm_model.score(factor_scores)
                
            except Exception as e2:
                print(f"Simplified HMM training also failed: {e2}")
                print("Falling back to KMeans clustering on factor scores")
                
                # Fallback to KMeans clustering
                from sklearn.cluster import KMeans
                kmeans = KMeans(n_clusters=self.n_states, random_state=42)
                regimes = kmeans.fit_predict(factor_scores)
                
                # Store KMeans model (no log likelihood)
                model['hmm_model'] = kmeans
                model['is_kmeans_fallback'] = True
                model['log_likelihood'] = -np.inf  # Default to worst likelihood
        
        return model, model['log_likelihood'], regimes
    
    def _predict(self, model, X_subset):
        """
        Make predictions with the trained HierarchicalHMM model.
        
        Parameters:
        -----------
        model : dict
            Trained model parameters
        X_subset : numpy.ndarray
            Feature data subset
            
        Returns:
        --------
        numpy.ndarray
            Predicted regimes
        """
        # Apply scaling if needed
        if model.get('scaler') is not None:
            X_scaled = model['scaler'].transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Transform to factor space
        factor_scores = model['factor_model'].transform(X_scaled)
        
        # Make prediction using the appropriate model
        if model.get('is_kmeans_fallback', False):
            # Using KMeans
            return model['hmm_model'].predict(factor_scores)
        else:
            # Using HMM
            return model['hmm_model'].predict(factor_scores)
    
    def get_factor_loadings(self, model):
        """
        Get the factor loadings showing how each original feature
        contributes to the latent factors.
        
        Parameters:
        -----------
        model : dict
            Trained model
            
        Returns:
        --------
        numpy.ndarray
            Factor loadings matrix
        """
        if model['factor_method'] == 'FactorAnalysis':
            # For Factor Analysis, components_ contains the loadings
            return model['factor_model'].components_
        elif model['factor_method'] == 'PCA':
            # For PCA, components_ contains the loadings
            return model['factor_model'].components_
        else:
            # Should not happen, but just in case
            return None
    
    def transform_to_factor_space(self, model, X_subset):
        """
        Transform data from feature space to factor space.
        
        Parameters:
        -----------
        model : dict
            Trained model
        X_subset : numpy.ndarray
            Feature data
            
        Returns:
        --------
        numpy.ndarray
            Data in factor space
        """
        # Apply scaling if needed
        if model.get('scaler') is not None:
            X_scaled = model['scaler'].transform(X_subset)
        else:
            X_scaled = X_subset
        
        # Transform to factor space
        return model['factor_model'].transform(X_scaled)
    
    def interpret_factors(self, model, feature_names):
        """
        Provide an interpretation of what each factor represents in terms of original features.
        
        Parameters:
        -----------
        model : dict
            Trained model
        feature_names : list
            Names of original features
            
        Returns:
        --------
        list
            Description of each factor
        """
        # Get factor loadings
        loadings = self.get_factor_loadings(model)
        
        if loadings is None:
            return ["Factor loadings not available"]
        
        # For each factor, find the features with highest absolute loadings
        interpretations = []
        
        for i, factor_loadings in enumerate(loadings):
            # Get indices of features sorted by absolute loading
            sorted_indices = np.argsort(np.abs(factor_loadings))[::-1]
            
            # Get top contributing features (at most 3 or all if fewer)
            top_n = min(3, len(feature_names))
            top_features = []
            
            for j in range(top_n):
                idx = sorted_indices[j]
                direction = "+" if factor_loadings[idx] > 0 else "-"
                contribution = np.abs(factor_loadings[idx])
                top_features.append(f"{direction}{feature_names[idx]} ({contribution:.2f})")
            
            interpretations.append(f"Factor {i+1}: {', '.join(top_features)}")
        
        return interpretations

### DNN with HMM (Will be explored in another notebook)

In [15]:
# import tensorflow as tf
# from tensorflow.keras.models import Sequential, Model
# from tensorflow.keras.layers import Dense, LSTM, Dropout, Input

# class DNNHMM:
#     """
#     DNN-HMM hybrid model for financial regime detection.
    
#     Uses a deep neural network to learn representations of financial data
#     and a Hidden Markov Model to detect regimes based on these representations.
#     """
    
#     def __init__(self, n_states=3, nn_type='feedforward', nn_layers=[64, 32], 
#                  activation='relu', dropout_rate=0.2, window_size=10):
#         """
#         Initialize the DNN-HMM model.
        
#         Parameters:
#         -----------
#         n_states : int
#             Number of hidden states (regimes) in the HMM
#         nn_type : str
#             Type of neural network ('feedforward' or 'lstm')
#         nn_layers : list
#             List of neurons in each hidden layer of the neural network
#         activation : str
#             Activation function for hidden layers
#         dropout_rate : float
#             Dropout rate for regularization
#         window_size : int
#             Window size for sequential data (used for LSTM)
#         """
#         self.n_states = n_states
#         self.nn_type = nn_type
#         self.nn_layers = nn_layers
#         self.activation = activation
#         self.dropout_rate = dropout_rate
#         self.window_size = window_size
#         self.feature_scaler = StandardScaler()
#         self.hmm = None
#         self.nn_model = None
        
#     def _build_feedforward_nn(self, input_dim, output_dim):
#         """Build a feedforward neural network for feature extraction."""
#         model = Sequential()
        
#         # Input layer
#         model.add(Dense(self.nn_layers[0], activation=self.activation, 
#                         input_shape=(input_dim,)))
#         model.add(Dropout(self.dropout_rate))
        
#         # Hidden layers
#         for units in self.nn_layers[1:]:
#             model.add(Dense(units, activation=self.activation))
#             model.add(Dropout(self.dropout_rate))
        
#         # Output layer - linear activation for feature extraction
#         model.add(Dense(output_dim, activation='linear', name='features'))
        
#         # Additional output for auxiliary task (e.g., return prediction)
#         # This helps with supervised pre-training
#         model.add(Dense(1, activation='linear', name='prediction'))
        
#         model.compile(optimizer='adam', loss={'prediction': 'mse', 'features': 'mse'})
#         return model
    
#     def _build_lstm_nn(self, input_dim, output_dim):
#         """Build an LSTM network for sequential feature extraction."""
#         inputs = Input(shape=(self.window_size, input_dim))
        
#         # LSTM layers
#         x = LSTM(self.nn_layers[0], return_sequences=len(self.nn_layers) > 1)(inputs)
#         x = Dropout(self.dropout_rate)(x)
        
#         # Additional LSTM layers
#         for i, units in enumerate(self.nn_layers[1:]):
#             return_sequences = i < len(self.nn_layers) - 2
#             x = LSTM(units, return_sequences=return_sequences)(x)
#             x = Dropout(self.dropout_rate)(x)
        
#         # Feature extraction layer
#         features = Dense(output_dim, activation='linear', name='features')(x)
        
#         # Prediction layer for auxiliary task
#         prediction = Dense(1, activation='linear', name='prediction')(x)
        
#         model = Model(inputs=inputs, outputs=[features, prediction])
#         model.compile(optimizer='adam', loss={'prediction': 'mse', 'features': 'mse'})
#         return model
    
#     def _prepare_lstm_data(self, X, y=None):
#         """Prepare data in the format required by LSTM (sequences)."""
#         n_samples = X.shape[0] - self.window_size + 1
#         n_features = X.shape[1]
        
#         # Create sequences
#         X_seq = np.zeros((n_samples, self.window_size, n_features))
#         for i in range(n_samples):
#             X_seq[i] = X[i:i+self.window_size]
        
#         if y is not None:
#             y_seq = y[self.window_size-1:]
#             return X_seq, y_seq
#         return X_seq
    
#     def pretrain_nn(self, X, y, validation_split=0.2, epochs=100, batch_size=32):
#         """
#         Pretrain the neural network using supervised learning.
        
#         Parameters:
#         -----------
#         X : array-like
#             Features
#         y : array-like
#             Target variable (e.g., future returns)
#         validation_split : float
#             Fraction of data to use for validation
#         epochs : int
#             Number of training epochs
#         batch_size : int
#             Batch size for training
#         """
#         # Scale features
#         X_scaled = self.feature_scaler.fit_transform(X)
        
#         # Set output dimension for features
#         feature_dim = min(X.shape[1], self.nn_layers[-1])
        
#         # Build neural network
#         if self.nn_type == 'feedforward':
#             self.nn_model = self._build_feedforward_nn(X.shape[1], feature_dim)
#             X_train, y_train = X_scaled, y
#         else:  # LSTM
#             self.nn_model = self._build_lstm_nn(X.shape[1], feature_dim)
#             X_train, y_train = self._prepare_lstm_data(X_scaled, y)
        
#         # Pretrain the network
#         self.nn_model.fit(
#             X_train, 
#             {'prediction': y_train, 'features': np.zeros((len(X_train), feature_dim))},  # Dummy features
#             epochs=epochs,
#             batch_size=batch_size,
#             validation_split=validation_split,
#             verbose=1
#         )
        
#         return self
    
#     def extract_features(self, X):
#         """
#         Extract features using the neural network.
        
#         Parameters:
#         -----------
#         X : array-like
#             Raw input features
            
#         Returns:
#         --------
#         array-like
#             Extracted features
#         """
#         if self.nn_model is None:
#             raise ValueError("Neural network model not trained. Call pretrain_nn first.")
        
#         # Scale features
#         X_scaled = self.feature_scaler.transform(X)
        
#         # Extract features using the trained model
#         if self.nn_type == 'feedforward':
#             # Create a new model that outputs only the features
#             feature_model = Model(
#                 inputs=self.nn_model.inputs,
#                 outputs=self.nn_model.get_layer('features').output
#             )
#             features = feature_model.predict(X_scaled)
#         else:  # LSTM
#             X_seq = self._prepare_lstm_data(X_scaled)
#             # Create a new model that outputs only the features
#             feature_model = Model(
#                 inputs=self.nn_model.inputs,
#                 outputs=self.nn_model.get_layer('features').output
#             )
#             features = feature_model.predict(X_seq)
        
#         return features
    
#     def fit(self, X, y=None, pretrain=True, pretrain_epochs=50):
#         """
#         Fit the DNN-HMM model.
        
#         Parameters:
#         -----------
#         X : array-like
#             Raw features
#         y : array-like, optional
#             Target for supervised pretraining
#         pretrain : bool
#             Whether to pretrain the neural network
#         pretrain_epochs : int
#             Number of epochs for pretraining
            
#         Returns:
#         --------
#         self
#         """
#         # Step 1: Pretrain the neural network if needed
#         if pretrain and y is not None:
#             self.pretrain_nn(X, y, epochs=pretrain_epochs)
        
#         # Step 2: Extract features using the neural network
#         features = self.extract_features(X)
        
#         # Step 3: Fit the HMM to the extracted features
#         self.hmm = hmm.GaussianHMM(
#             n_components=self.n_states,
#             covariance_type="full",
#             n_iter=1000,
#             tol=1e-8,
#             random_state=42
#         )
#         self.hmm.fit(features)
        
#         return self
    
#     def predict_regimes(self, X):
#         """
#         Predict regimes using the fitted DNN-HMM model.
        
#         Parameters:
#         -----------
#         X : array-like
#             Raw features
            
#         Returns:
#         --------
#         array-like
#             Predicted regime for each time point
#         """
#         if self.hmm is None:
#             raise ValueError("Model not fitted. Call fit first.")
        
#         # Extract features using the neural network
#         features = self.extract_features(X)
        
#         # Predict hidden states (regimes)
#         regimes = self.hmm.predict(features)
        
#         # Ensure regimes start from 0 for consistency
#         if self.nn_type == 'lstm':
#             # For LSTM, we lose the first (window_size-1) points due to sequence formation
#             padding = np.full(self.window_size-1, -1)  # Use -1 to indicate no prediction
#             regimes = np.concatenate([padding, regimes])
        
#         return regimes
    
#     def score_samples(self, X):
#         """
#         Compute log-likelihood of each sample.
        
#         Parameters:
#         -----------
#         X : array-like
#             Raw features
            
#         Returns:
#         --------
#         array-like
#             Log-likelihood for each time point
#         """
#         if self.hmm is None:
#             raise ValueError("Model not fitted. Call fit first.")
        
#         # Extract features using the neural network
#         features = self.extract_features(X)
        
#         # Compute log-likelihood of each sample
#         log_likelihood, _ = self.hmm.score_samples(features)
        
#         return log_likelihood

# # 3. Train DNN-HMM model
# def train_dnn_hmm(features, target, nn_type='feedforward', n_states=3):
#     # Initialize and fit the model
#     model = DNNHMM(
#         n_states=n_states,
#         nn_type=nn_type,
#         nn_layers=[128, 64, 32],
#         activation='relu',
#         dropout_rate=0.3,
#         window_size=20
#     )
    
#     model.fit(features.values, target.values, pretrain=True, pretrain_epochs=100)
    
#     # Predict regimes
#     regimes = model.predict_regimes(features.values)
    
#     return model, regimes

# # 4. Evaluate and visualize results
# def evaluate_regimes(data, regimes, dates):
#     # Calculate cumulative returns for each regime
#     returns = data['Adj Close'].pct_change().dropna()
    
#     # Ensure the arrays have the same length
#     min_length = min(len(returns), len(regimes))
#     returns = returns.iloc[:min_length]
#     regimes = regimes[:min_length]
    
#     # Create DataFrame with returns and regimes
#     results = pd.DataFrame({
#         'returns': returns.values,
#         'regime': regimes
#     }, index=returns.index[:min_length])
    
#     # Calculate stats for each regime
#     regime_stats = {}
#     for regime in np.unique(regimes):
#         if regime == -1:  # Skip padding values from LSTM
#             continue
            
#         regime_returns = results[results['regime'] == regime]['returns']
        
#         if len(regime_returns) > 0:
#             regime_stats[regime] = {
#                 'count': len(regime_returns),
#                 'mean_return': regime_returns.mean() * 252,  # Annualized return
#                 'volatility': regime_returns.std() * np.sqrt(252),  # Annualized volatility
#                 'sharpe': (regime_returns.mean() / regime_returns.std()) * np.sqrt(252),
#                 'skewness': pd.Series(regime_returns).skew(),
#                 'kurtosis': pd.Series(regime_returns).kurt(),
#                 'max_drawdown': calculate_max_drawdown(regime_returns),
#                 'win_rate': (regime_returns > 0).sum() / len(regime_returns)
#             }
    
#     # Visualize the regimes
#     plt.figure(figsize=(15, 10))
    
#     # Plot 1: Asset price with colored regimes
#     plt.subplot(2, 1, 1)
    
#     colors = ['green', 'blue', 'red']
    
#     # Plot the asset price
#     asset_price = data['Adj Close']
    
#     for regime in range(len(colors)):
#         if regime not in results['regime'].unique():
#             continue
            
#         regime_data = results[results['regime'] == regime]
        
#         # Extract the dates for this regime
#         regime_dates = regime_data.index
        
#         # Plot asset price for these dates using the regime color
#         regime_prices = asset_price.loc[regime_dates]
#         plt.plot(regime_dates, regime_prices, color=colors[regime])
    
#     plt.title('Asset Price Colored by Market Regime')
#     plt.ylabel('Price')
#     plt.grid(True, alpha=0.3)
    
#     # Plot 2: Return distributions by regime
#     plt.subplot(2, 1, 2)
    
#     for regime in range(len(colors)):
#         if regime not in results['regime'].unique():
#             continue
            
#         regime_returns = results[results['regime'] == regime]['returns']
        
#         if len(regime_returns) > 0:
#             plt.hist(regime_returns, bins=50, alpha=0.5, color=colors[regime], 
#                      label=f'Regime {regime}')
    
#     plt.axvline(x=0, color='black', linestyle='-', alpha=0.5)
#     plt.title('Return Distributions by Regime')
#     plt.xlabel('Daily Returns')
#     plt.ylabel('Frequency')
#     plt.legend()
#     plt.grid(True, alpha=0.3)
    
#     plt.tight_layout()
#     plt.show()
    
#     return regime_stats

# # Helper function for max drawdown calculation
# def calculate_max_drawdown(returns_series):
#     cumulative = (1 + returns_series).cumprod()
#     running_max = cumulative.cummax()
#     drawdowns = (cumulative / running_max) - 1
#     return drawdowns.min()

# # Main execution
# if __name__ == "__main__":
#     # Get data
#     data = get_data(ticker='SPY', start='2010-01-01', end='2023-12-31')
    
#     # Create features
#     features, target, dates = create_features(data)
    
#     # Train DNN-HMM model
#     model, regimes = train_dnn_hmm(features, target, nn_type='lstm', n_states=3)
    
#     # Evaluate results
#     regime_stats = evaluate_regimes(data, regimes, dates)
    
#     # Print statistics for each regime
#     for regime, stats in regime_stats.items():
#         print(f"\nRegime {regime} statistics:")
#         for stat_name, value in stats.items():
#             print(f"  {stat_name}: {value:.4f}" if isinstance(value, float) else f"  {stat_name}: {value}")

### LeaderBoard

In [16]:
class HMMLeaderboard:
    """
    Class for creating and managing leaderboards of HMM models.
    Consolidates results from multiple HMM model types for comparison.
    """
    
    def __init__(self):
        """Initialize the HMMLeaderboard."""
        self.leaderboard = None
    
    def create_leaderboard(self, hmm_models_list, sort_by='silhouette_score', ascending=False, top_n=None):
        """
        Create a consolidated leaderboard for multiple HMM models.
        
        Parameters:
        -----------
        hmm_models_list : list
            List of trained HMM model instances
        sort_by : str
            Metric to sort by ('silhouette_score' or 'log_likelihood')
        ascending : bool
            Sort in ascending (True) or descending (False) order
        top_n : int or None
            Number of top models to include, if None shows all
            
        Returns:
        --------
        pandas.DataFrame
            Leaderboard with all models sorted by specified metric
        """
        # Collect all model information into a single list
        all_models_info = []
        
        for model_obj in hmm_models_list:
            # Extract information from each model's models_info dictionary
            for model_id, model_info in model_obj.models_info.items():
                # Get base information
                info_dict = {
                    'model_id': model_id,
                    'model_type': model_info['model_type'],
                    'scaling': model_info['scaling'],
                    'features': '+'.join(model_info['features']),
                    'n_features': len(model_info['features']),
                    'silhouette_score': model_info.get('silhouette_score'),
                    'log_likelihood': model_info.get('log_likelihood')
                }
                
                # Add specialized model information if available
                if 'n_factors' in model_info:
                    info_dict['n_factors'] = model_info['n_factors']
                
                if 'external_factor' in model_info:
                    info_dict['external_factor'] = model_info['external_factor']
                
                if hasattr(model_obj, 'n_mixtures'):
                    info_dict['n_mixtures'] = model_obj.n_mixtures
                    
                if hasattr(model_obj, 'window_size'):
                    info_dict['window_size'] = model_obj.window_size
                    info_dict['step_size'] = model_obj.step_size
                
                # Add to collection
                all_models_info.append(info_dict)
        
        # Create DataFrame
        if not all_models_info:
            self.leaderboard = pd.DataFrame()  # Empty DataFrame if no models
            return self.leaderboard
        
        leaderboard_df = pd.DataFrame(all_models_info)
        
        # Sort by specified metric
        if sort_by in leaderboard_df.columns:
            # Special handling for NaN values - put them at the end
            leaderboard_df = leaderboard_df.sort_values(
                by=sort_by, 
                ascending=ascending,
                na_position='last'
            ).reset_index(drop=True)
        
        # Limit to top N if specified
        if top_n is not None and top_n < len(leaderboard_df):
            leaderboard_df = leaderboard_df.iloc[:top_n]
        
        # Store and return the leaderboard
        self.leaderboard = leaderboard_df
        return self.leaderboard
    
    def get_top_models(self, n=5):
        """
        Get the top N models from the leaderboard.
        
        Parameters:
        -----------
        n : int
            Number of top models to return
            
        Returns:
        --------
        pandas.DataFrame
            Top N models from the leaderboard
        """
        if self.leaderboard is None or self.leaderboard.empty:
            return pd.DataFrame()
        
        return self.leaderboard.head(n)
    
    def get_model_by_id(self, model_id):
        """
        Get a specific model from the leaderboard by ID.
        
        Parameters:
        -----------
        model_id : str
            ID of the model to retrieve
            
        Returns:
        --------
        pandas.Series or None
            Model information if found, None otherwise
        """
        if self.leaderboard is None or self.leaderboard.empty:
            return None
        
        matching_models = self.leaderboard[self.leaderboard['model_id'] == model_id]
        if not matching_models.empty:
            return matching_models.iloc[0]
        
        return None
    
    def filter_leaderboard(self, model_type=None, scaling=None, min_silhouette=None, min_likelihood=None):
        """
        Filter the leaderboard based on various criteria.
        
        Parameters:
        -----------
        model_type : str or None
            Filter by model type
        scaling : str or None
            Filter by scaling type ('robust' or 'none')
        min_silhouette : float or None
            Minimum silhouette score
        min_likelihood : float or None
            Minimum log likelihood
            
        Returns:
        --------
        pandas.DataFrame
            Filtered leaderboard
        """
        if self.leaderboard is None or self.leaderboard.empty:
            return pd.DataFrame()
        
        filtered = self.leaderboard.copy()
        
        if model_type is not None:
            filtered = filtered[filtered['model_type'] == model_type]
        
        if scaling is not None:
            filtered = filtered[filtered['scaling'] == scaling]
        
        if min_silhouette is not None:
            filtered = filtered[
                (filtered['silhouette_score'] >= min_silhouette) | 
                (filtered['silhouette_score'].isna())
            ]
        
        if min_likelihood is not None:
            filtered = filtered[
                (filtered['log_likelihood'] >= min_likelihood) | 
                (filtered['log_likelihood'].isna())
            ]
        
        return filtered
    
    def export_to_csv(self, filename='hmm_leaderboard.csv'):
        """
        Export the leaderboard to a CSV file.
        
        Parameters:
        -----------
        filename : str
            Output filename
            
        Returns:
        --------
        bool
            Success indicator
        """
        if self.leaderboard is None or self.leaderboard.empty:
            print("No leaderboard data to export.")
            return False
        
        try:
            self.leaderboard.to_csv(filename, index=False)
            print(f"Leaderboard exported to {filename}")
            return True
        except Exception as e:
            print(f"Error exporting leaderboard: {e}")
            return False

### Visualize

In [17]:
class HMMVisualizer:
    """
    Class for visualizing HMM model results and performance comparisons.
    Provides tools for regime visualization, heatmaps, and feature importance analysis.
    """
    
    def __init__(self, hmm_models_list=None):
        """
        Initialize the HMMVisualizer.
        
        Parameters:
        -----------
        hmm_models_list : list, optional
            List of trained HMM model instances
        """
        self.hmm_models_list = hmm_models_list if hmm_models_list is not None else []
        self.model_obj_map = {}
        
        # Create a mapping from model_id to the corresponding model object
        if self.hmm_models_list:
            for model_obj in self.hmm_models_list:
                for model_id in model_obj.models_info.keys():
                    self.model_obj_map[model_id] = model_obj
    
    def add_models(self, hmm_models_list):
        """
        Add more HMM models to the visualizer.
        
        Parameters:
        -----------
        hmm_models_list : list
            List of trained HMM model instances
        """
        self.hmm_models_list.extend(hmm_models_list)
        
        # Update the model object mapping
        for model_obj in hmm_models_list:
            for model_id in model_obj.models_info.keys():
                self.model_obj_map[model_id] = model_obj
    
    def visualize_top_models(self, leaderboard, X_data, price_data, top_n=5, figsize=(15, 8)):
        """
        Visualize regimes for top models in the leaderboard.
        
        Parameters:
        -----------
        leaderboard : pandas.DataFrame
            Leaderboard with sorted models
        X_data : pandas.DataFrame
            Feature data used for training
        price_data : pandas.Series
            Price data for visualization
        top_n : int
            Number of top models to visualize
        figsize : tuple
            Figure size for plots
            
        Returns:
        --------
        None
        """
        if leaderboard.empty:
            print("Leaderboard is empty. Nothing to visualize.")
            return
        
        # Limit to top N models
        top_models = leaderboard.head(top_n)
        
        # Define colors for regimes
        colors = ['green', 'blue', 'red', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'brown']
        
        # Plot each top model
        for index, row in top_models.iterrows():
            model_id = row['model_id']
            model_obj = self.model_obj_map.get(model_id)
            
            if model_obj is None:
                print(f"Warning: Model object for {model_id} not found.")
                continue
            
            # Get model information and regimes
            model_info = model_obj.models_info[model_id]
            regimes = model_info['regimes']
            
            # Create DataFrame with price and regimes
            df = pd.DataFrame({
                'price': price_data,
                'regime': pd.Series(regimes, index=price_data.index)
            })
            
            # Create plot
            plt.figure(figsize=figsize)
            
            # Plot price
            plt.plot(df.index, df['price'], color='black', linewidth=1.5, label='Price')
            
            # Get number of states/regimes
            n_states = model_obj.n_states
            
            # Add regime backgrounds
            for j in range(n_states):
                regime_data = df[df['regime'] == j]
                regime_start_dates = []
                regime_end_dates = []
                
                # Find continuous regime periods
                if not regime_data.empty:
                    current_start = regime_data.index[0]
                    prev_date = regime_data.index[0]
                    
                    for date in regime_data.index[1:]:
                        # Check if dates are consecutive
                        if (date - prev_date).days > 5:  # Allow small gaps
                            regime_start_dates.append(current_start)
                            regime_end_dates.append(prev_date)
                            current_start = date
                        prev_date = date
                    
                    # Add the last period
                    if not regime_data.empty:
                        regime_start_dates.append(current_start)
                        regime_end_dates.append(regime_data.index[-1])
                
                # Color the background for each period
                for start, end in zip(regime_start_dates, regime_end_dates):
                    plt.axvspan(start, end, alpha=0.2, color=colors[j % len(colors)])
            
            # Add regime transitions as vertical lines
            transitions = []
            for k in range(1, len(regimes)):
                if regimes[k] != regimes[k-1]:
                    transitions.append(df.index[k])
            
            for transition in transitions:
                plt.axvline(x=transition, linestyle='--', color='gray', alpha=0.7)
            
            # Add chart title
            title = f"Model #{index+1}: {row['model_type']}"
            if 'n_factors' in row:
                title += f" (Factors: {row['n_factors']})"
            if 'external_factor' in row:
                title += f" (Ext. Factor: {row['external_factor']})"
            if 'n_mixtures' in row:
                title += f" (Mixtures: {row['n_mixtures']})"
            if 'window_size' in row:
                title += f" (Window: {row['window_size']})"
                
            title += f"\nFeatures: {row['features']}, Scaling: {row['scaling']}"
            title += f"\nSilhouette: {row['silhouette_score']:.4f}, Log Likelihood: {row['log_likelihood']:.2f}"
            
            plt.title(title, fontsize=12)
            plt.xlabel('Date', fontsize=12)
            plt.ylabel('Price', fontsize=12)
            
            # Create legend for regimes
            legend_elements = []
            
            # Calculate statistics for each regime
            for j in range(n_states):
                regime_data = df[df['regime'] == j]
                if not regime_data.empty:
                    # Calculate regime statistics
                    returns = regime_data['price'].pct_change().dropna()
                    if len(returns) > 1:
                        mean_return = returns.mean() * 252 * 100  # Annualized
                        volatility = returns.std() * np.sqrt(252) * 100  # Annualized
                        sharpe = mean_return / volatility if volatility > 0 else 0
                        pct = len(regime_data) / len(df) * 100
                        
                        label = f'Regime {j}: Return={mean_return:.2f}%, Vol={volatility:.2f}%, Sharpe={sharpe:.2f} ({pct:.1f}%)'
                        legend_elements.append(plt.Rectangle((0,0), 1, 1, fc=colors[j % len(colors)], alpha=0.2, label=label))
            
            plt.legend(handles=legend_elements, loc='upper left', fontsize=10)
            plt.grid(True, alpha=0.3)
            
            # Format x-axis with dates
            plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
            plt.gca().xaxis.set_major_locator(mdates.YearLocator())
            plt.gcf().autofmt_xdate()
            
            plt.tight_layout()
            plt.show()
    
    def create_heatmap_leaderboard(self, leaderboard, metric='silhouette_score', figsize=(12, 8)):
        """
        Create a heatmap visualization of model performance across different 
        model types and feature combinations.
        
        Parameters:
        -----------
        leaderboard : pandas.DataFrame
            Leaderboard with model information
        metric : str
            Metric to visualize ('silhouette_score' or 'log_likelihood')
        figsize : tuple
            Figure size for plot
            
        Returns:
        --------
        None
        """
        if leaderboard.empty:
            print("Empty leaderboard, nothing to visualize.")
            return
        
        # Ensure the metric exists
        if metric not in leaderboard.columns:
            print(f"Metric {metric} not found in leaderboard.")
            return
        
        # Create pivot table: model_type vs features
        pivot = pd.pivot_table(
            leaderboard,
            values=metric,
            index='model_type',
            columns='features',
            aggfunc='max'  # Use max value when multiple models exist
        )
        
        # Create figure
        plt.figure(figsize=figsize)
        
        # Define colormap (blue for lower values, red for higher values)
        cmap = LinearSegmentedColormap.from_list(
            'blue_white_red', ['blue', 'white', 'red'], N=100
        )
        if metric == 'log_likelihood':
            # For log likelihood, typically negative values, reversed colormap
            cmap = LinearSegmentedColormap.from_list(
                'red_white_blue', ['red', 'white', 'blue'], N=100
            )
        
        # Create heatmap
        sns.heatmap(
            pivot, 
            annot=True, 
            cmap=cmap,
            linewidths=0.5, 
            fmt='.2f',
            vmin=leaderboard[metric].min() if leaderboard[metric].min() is not None else None,
            vmax=leaderboard[metric].max() if leaderboard[metric].max() is not None else None,
            center=0 if metric == 'silhouette_score' else None,
            cbar_kws={'label': metric}
        )
        
        plt.title(f'Model Performance by {metric}', fontsize=14)
        plt.xlabel('Features')
        plt.ylabel('Model Type')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    
    def analyze_feature_importance(self, leaderboard, metric='silhouette_score', top_percentile=25, visualize=True):
        """
        Analyze which features are most important across top-performing models.
        
        Parameters:
        -----------
        leaderboard : pandas.DataFrame
            Leaderboard with model information
        metric : str
            Metric to use for ranking ('silhouette_score' or 'log_likelihood')
        top_percentile : int
            Percentile of top models to consider (e.g., 25 = top 25%)
        visualize : bool
            Whether to create visualization of feature importance
            
        Returns:
        --------
        pandas.DataFrame
            Feature importance analysis
        """
        if leaderboard.empty:
            print("Empty leaderboard, nothing to analyze.")
            return pd.DataFrame()
        
        # Sort by metric
        sorted_board = leaderboard.sort_values(
            by=metric, 
            ascending=False, 
            na_position='last'
        )
        
        # Get the top N models
        top_n = max(1, int(len(sorted_board) * top_percentile / 100))
        top_models = sorted_board.head(top_n)
        
        # Extract all features used in top models
        all_features = set()
        
        for _, row in top_models.iterrows():
            features = row['features'].split('+')
            all_features.update(features)
        
        # Count feature occurrences in top models
        feature_counts = {feature: 0 for feature in all_features}
        feature_performance = {feature: [] for feature in all_features}
        
        for _, row in top_models.iterrows():
            features = row['features'].split('+')
            for feature in features:
                feature_counts[feature] += 1
                if not pd.isna(row[metric]):
                    feature_performance[feature].append(row[metric])
        
        # Calculate average performance for each feature
        feature_avg_performance = {}
        for feature, values in feature_performance.items():
            if values:
                feature_avg_performance[feature] = sum(values) / len(values)
            else:
                feature_avg_performance[feature] = None
        
        # Create DataFrame
        analysis = pd.DataFrame({
            'feature': list(all_features),
            'count': [feature_counts[f] for f in all_features],
            'frequency': [feature_counts[f] / top_n for f in all_features],
            f'avg_{metric}': [feature_avg_performance[f] for f in all_features]
        })
        
        # Sort by frequency and average performance
        analysis = analysis.sort_values(
            by=['frequency', f'avg_{metric}'], 
            ascending=[False, False if metric == 'silhouette_score' else True]
        ).reset_index(drop=True)
        
        # Visualize feature importance if requested
        if visualize and not analysis.empty:
            self._plot_feature_importance(analysis, metric, top_percentile)
        
        return analysis
    
    def _plot_feature_importance(self, analysis, metric, top_percentile):
        """
        Plot feature importance analysis.
        
        Parameters:
        -----------
        analysis : pandas.DataFrame
            Feature importance analysis
        metric : str
            Performance metric used
        top_percentile : int
            Percentile of top models considered
            
        Returns:
        --------
        None
        """
        # Create figure with two subplots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot feature frequency
        sns.barplot(
            x='frequency', 
            y='feature', 
            data=analysis.sort_values('frequency', ascending=False), 
            ax=ax1,
            palette='viridis'
        )
        ax1.set_title(f'Feature Frequency in Top {top_percentile}% of Models', fontsize=12)
        ax1.set_xlabel('Frequency', fontsize=10)
        ax1.set_ylabel('Feature', fontsize=10)
        ax1.grid(axis='x', alpha=0.3)
        
        # Plot average performance
        sns.barplot(
            x=f'avg_{metric}', 
            y='feature', 
            data=analysis.sort_values(f'avg_{metric}', ascending=False if metric == 'log_likelihood' else False), 
            ax=ax2,
            palette='magma'
        )
        ax2.set_title(f'Average {metric} by Feature', fontsize=12)
        ax2.set_xlabel(f'Average {metric}', fontsize=10)
        ax2.set_ylabel('')  # No label needed for y-axis
        ax2.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
    def plot_regime_distributions(self, model_id, X_data, price_data, figsize=(15, 10)):
        """
        Plot distribution of returns for each regime.
        
        Parameters:
        -----------
        model_id : str
            ID of the model to visualize
        X_data : pandas.DataFrame
            Feature data
        price_data : pandas.Series
            Price data for return calculation
        figsize : tuple
            Figure size for plots
            
        Returns:
        --------
        None
        """
        # Find the model object
        model_obj = self.model_obj_map.get(model_id)
        if model_obj is None:
            print(f"Model with ID {model_id} not found.")
            return
        
        # Get model information
        model_info = model_obj.models_info[model_id]
        regimes = model_info['regimes']
        n_states = model_obj.n_states
        
        # Create returns series
        returns = price_data.pct_change().dropna()
        
        # Create DataFrame with returns and regimes
        df = pd.DataFrame({
            'returns': returns,
            'regime': pd.Series(regimes, index=returns.index)
        })
        
        # Define colors for regimes
        colors = ['green', 'blue', 'red', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'brown']
        
        # Create figure
        fig, axes = plt.subplots(n_states, 2, figsize=figsize)
        if n_states == 1:
            axes = axes.reshape(1, 2)
        
        # Plot each regime
        for i in range(n_states):
            regime_data = df[df['regime'] == i]
            
            if not regime_data.empty:
                # Calculate statistics
                mean = regime_data['returns'].mean() * 100
                std = regime_data['returns'].std() * 100
                skew = regime_data['returns'].skew()
                kurt = regime_data['returns'].kurtosis()
                
                # Plot histogram
                ax = axes[i, 0]
                sns.histplot(regime_data['returns'] * 100, kde=True, ax=ax, color=colors[i % len(colors)])
                ax.axvline(mean, color='red', linestyle='--', alpha=0.8)
                ax.set_title(f'Regime {i} Return Distribution')
                ax.set_xlabel('Returns (%)')
                ax.set_ylabel('Frequency')
                
                # Add statistics as text
                stats_text = (f"Mean: {mean:.2f}%\nStd: {std:.2f}%\n"
                              f"Skew: {skew:.2f}\nKurtosis: {kurt:.2f}\n"
                              f"Count: {len(regime_data)}")
                ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, 
                        verticalalignment='top', horizontalalignment='right',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                # Plot time series
                ax = axes[i, 1]
                ax.scatter(regime_data.index, regime_data['returns'] * 100, 
                          color=colors[i % len(colors)], alpha=0.6, s=15)
                ax.axhline(0, color='black', linestyle='-', alpha=0.3)
                ax.set_title(f'Regime {i} Returns Over Time')
                ax.set_xlabel('Date')
                ax.set_ylabel('Returns (%)')
                
                # Format x-axis with dates
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
                ax.xaxis.set_major_locator(mdates.YearLocator())
                plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
            else:
                # If no data for this regime
                for j in range(2):
                    axes[i, j].text(0.5, 0.5, f"No data for Regime {i}", 
                                   ha='center', va='center', fontsize=12)
                    axes[i, j].set_title(f'Regime {i}')
        
        plt.tight_layout()
        plt.show()
        
    def plot_model_comparison(self, leaderboard, metric='silhouette_score', groupby='model_type', top_n=10, figsize=(12, 6)):
        """
        Plot comparison of model performance grouped by specified column.
        
        Parameters:
        -----------
        leaderboard : pandas.DataFrame
            Leaderboard with model information
        metric : str
            Metric to visualize ('silhouette_score' or 'log_likelihood')
        groupby : str
            Column to group by ('model_type', 'scaling', etc.)
        top_n : int
            Number of top models to include
        figsize : tuple
            Figure size for plot
            
        Returns:
        --------
        None
        """
        if leaderboard.empty:
            print("Empty leaderboard, nothing to visualize.")
            return
        
        # Ensure the metric and groupby column exist
        if metric not in leaderboard.columns:
            print(f"Metric {metric} not found in leaderboard.")
            return
        
        if groupby not in leaderboard.columns:
            print(f"Column {groupby} not found in leaderboard.")
            return
        
        # Get top N rows
        top_models = leaderboard.head(top_n)
        
        # Create figure
        plt.figure(figsize=figsize)
        
        # Create the plot
        ax = sns.barplot(
            x=groupby, 
            y=metric, 
            data=top_models,
            palette='viridis',
            errorbar=None
        )
        
        # Customize the plot
        plt.title(f'Top {top_n} Models by {metric} Grouped by {groupby}', fontsize=14)
        plt.xlabel(groupby.replace('_', ' ').title(), fontsize=12)
        plt.ylabel(metric.replace('_', ' ').title(), fontsize=12)
        
        # Add value labels on top of bars
        for i, p in enumerate(ax.patches):
            ax.annotate(f'{p.get_height():.3f}', 
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='bottom', fontsize=9, rotation=45)
        
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## Run

In [18]:
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Define classes directly or import from current namespace
from __main__ import StandardHMM, SkewTHMM, RollingWindowHMM, NonHomogeneousHMM, HMMWithGMM, HierarchicalHMM, HMMLeaderboard, HMMVisualizer

def run_all_hmm_models(X_train, X_train_scaled, features, n_states=3):
    """
    Train and evaluate all types of HMM models.
    
    Parameters:
    -----------
    X_train : pandas.DataFrame
        Raw training data
    X_train_scaled : pandas.DataFrame
        Scaled training data
    features : list
        List of feature combinations to evaluate
    n_states : int
        Number of hidden states/regimes
        
    Returns:
    --------
    list
        List of trained HMM model objects
    """
    # List to store trained models
    hmm_models = []
    
    # 1. Standard Gaussian HMM
    print("\n======= Training Standard Gaussian HMM Models =======")
    standard_hmm = StandardHMM(n_states=n_states)
    standard_hmm.train_all_models(X_train, X_train_scaled, features)
    hmm_models.append(standard_hmm)
    print(f"Trained {len(standard_hmm.models_info)} Standard HMM models")
    
    # 2. SkewT-HMM
    print("\n======= Training SkewT-HMM Models =======")
    skewt_hmm = SkewTHMM(n_states=n_states)
    skewt_hmm.train_all_models(X_train, X_train_scaled, features)
    hmm_models.append(skewt_hmm)
    print(f"Trained {len(skewt_hmm.models_info)} SkewT-HMM models")
    
    # 3. Rolling Window HMM (window size 730, step size 21)
    print("\n======= Training Rolling Window HMM Models =======")
    rolling_hmm = RollingWindowHMM(n_states=n_states, window_size=730, step_size=21)
    rolling_hmm.train_all_models(X_train, X_train_scaled, features)
    hmm_models.append(rolling_hmm)
    print(f"Trained {len(rolling_hmm.models_info)} Rolling Window HMM models")
    
    # 4. Non-Homogeneous HMM (using each feature as external factor)
    print("\n======= Training Non-Homogeneous HMM Models =======")
    nonhomo_hmm = NonHomogeneousHMM(n_states=n_states)
    # For non-homogeneous HMM, we'll use each feature individually as an external factor
    external_factors = list(X_train.columns)
    nonhomo_hmm.train_all_models(X_train, X_train_scaled, features, external_factors)
    hmm_models.append(nonhomo_hmm)
    print(f"Trained {len(nonhomo_hmm.models_info)} Non-Homogeneous HMM models")
    
    # 5. HMM with GMM (3 mixtures)
    print("\n======= Training HMM with GMM Models =======")
    gmm_hmm = HMMWithGMM(n_states=n_states, n_mixtures=3)
    gmm_hmm.train_all_models(X_train, X_train_scaled, features)
    hmm_models.append(gmm_hmm)
    print(f"Trained {len(gmm_hmm.models_info)} HMM with GMM models")
    
    # 6. Hierarchical HMM (with factor analysis)
    # Note: Only use feature combinations with at least 2 features
    print("\n======= Training Hierarchical HMM Models =======")
    hierarchical_hmm = HierarchicalHMM(n_states=n_states)
    hierarchical_hmm.train_all_models(X_train, X_train_scaled, features)
    hmm_models.append(hierarchical_hmm)
    print(f"Trained {len(hierarchical_hmm.models_info)} Hierarchical HMM models")
    
    return hmm_models


def evaluate_and_visualize(hmm_models, X_train, train_price, X_test=None, test_price=None, top_n=10, save_leaderboard=True):
    """
    Create leaderboard and visualizations for trained HMM models.
    """
    # Create leaderboard
    print("\n======= Creating Model Leaderboard =======")
    leaderboard_creator = HMMLeaderboard()
    leaderboard = leaderboard_creator.create_leaderboard(hmm_models, sort_by='silhouette_score')
    
    # Check if leaderboard is empty or malformed
    if leaderboard.empty:
        print("Warning: Leaderboard is empty. No models were successfully trained or evaluated.")
        return leaderboard, None
    
    # Check if expected columns exist
    expected_cols = ['model_type', 'scaling', 'features', 'silhouette_score', 'log_likelihood']
    missing_cols = [col for col in expected_cols if col not in leaderboard.columns]
    
    if missing_cols:
        print(f"Warning: Leaderboard is missing expected columns: {missing_cols}")
        print("Available columns:", leaderboard.columns.tolist())
        
        # Display what we can
        display_cols = [col for col in expected_cols if col in leaderboard.columns]
        if display_cols:
            print(f"\nDisplaying leaderboard with available columns:")
            print(leaderboard.head(top_n)[display_cols])
        else:
            print("\nLeaderboard structure:")
            print(leaderboard.head())
    
    # Create visualizer
    print("\n======= Creating Visualizations =======")
    visualizer = HMMVisualizer(hmm_models)
    
    # Display top models info
    print("\n======= Top Models by Silhouette Score =======")
    top_models = leaderboard.sort_values('silhouette_score', ascending=False).head(top_n)
    display_cols = [col for col in ['model_id', 'model_type', 'features', 'scaling', 'silhouette_score'] 
                   if col in leaderboard.columns]
    print(top_models[display_cols])
    
    # Visualize top models on training data
    print("\n======= Visualizing Top Models (Training Data) =======")
    visualizer.visualize_top_models(top_models, X_train, train_price, top_n=min(5, top_n))
    
    # Create heatmap leaderboard
    print("\n======= Creating Heatmap Leaderboard =======")
    visualizer.create_heatmap_leaderboard(leaderboard)
    
    # Analyze feature importance
    print("\n======= Analyzing Feature Importance =======")
    feature_importance = visualizer.analyze_feature_importance(leaderboard)
    
    # Visualize on test data if available
    if X_test is not None and test_price is not None:
        print("\n======= Visualizing Top Models (Test Data) =======")
        # We'll just use the top 3 models for test visualization to keep it concise
        test_top_n = min(3, top_n)
        visualizer.visualize_top_models(top_models.head(test_top_n), X_test, test_price, top_n=test_top_n)
    
    # Save leaderboard if requested
    if save_leaderboard:
        leaderboard_creator.export_to_csv('hmm_leaderboard.csv')
    
    return leaderboard, visualizer


def run_hmm_evaluation(X_train, X_train_scaled, train_price, features, X_test=None, test_price=None, 
                       n_states=3, top_n=10):
    """
    Complete HMM model evaluation pipeline using existing data and feature combinations.
    
    Parameters:
    -----------
    X_train : pandas.DataFrame
        Raw training data
    X_train_scaled : pandas.DataFrame
        Scaled training data
    train_price : pandas.Series
        Price data for training period
    features : list
        List of feature combinations to evaluate
    X_test : pandas.DataFrame, optional
        Test data
    test_price : pandas.Series, optional
        Price data for test period
    n_states : int
        Number of hidden states/regimes
    top_n : int
        Number of top models to display
        
    Returns:
    --------
    dict
        Results dictionary with models, leaderboard, and visualizer
    """
    print(f"Starting HMM model evaluation with {len(features)} feature combinations")
    
    # Train all HMM models
    hmm_models = run_all_hmm_models(X_train, X_train_scaled, features, n_states=n_states)
    
    # Evaluate and visualize results
    leaderboard, visualizer = evaluate_and_visualize(
        hmm_models, 
        X_train, 
        train_price, 
        X_test, 
        test_price, 
        top_n=top_n
    )
    
    print("\n======= Analysis Complete =======")
    
    return {
        'hmm_models': hmm_models,
        'leaderboard': leaderboard,
        'visualizer': visualizer
    }


# Example usage
if __name__ == "__main__":
    # This assumes X_train, X_train_scaled, train_price, and features are already defined
    # in the global scope
    
    # Run the evaluation
    results = run_hmm_evaluation(
        X_train=X_train,  
        X_train_scaled=X_train_scaled, 
        train_price=train_price,
        features=features,  # Using existing feature combinations
        X_test=X_test if 'X_test' in globals() else None,
        test_price=test_price if 'test_price' in globals() else None,
        n_states=3
    )

Starting HMM model evaluation with 7 feature combinations

======= Training Standard Gaussian HMM Models =======
Error loading checkpoint: unhashable type: 'list'. Starting fresh.

Processing batch 1/1 with 14 tasks
Training StandardHMM models with none scaling


Training models (none scaling):   0%|                                                            | 0/7 [00:00<?, ?it/s]

  Training with X_subset shape: (2058, 1)


Training models (none scaling):  29%|██████████████▊                                     | 2/7 [00:00<00:01,  3.17it/s]

  Training with X_subset shape: (2058, 1)
  Training with X_subset shape: (2058, 1)
  Training with X_subset shape: (2058, 2)


...

Training models (none scaling): 100%|████████████████████████████████████████████████████| 7/7 [02:45<00:00, 23.71s/it]


Training RollingHMM models with robust scaling


Training models (robust scaling): 100%|██████████████████████████████████████████████████| 7/7 [02:51<00:00, 24.50s/it]


Saved checkpoint after batch 1. 14 features processed.
Trained 14 Rolling Window HMM models

======= Training Non-Homogeneous HMM Models =======

Training with external factor: Range
  Using none scaling
  Using robust scaling


KeyboardInterrupt: 